# 🍀 はじめに 🍀
# 😊メンバーシップにご参加の皆さん！いつも温かい応援ありがとうございます！
### 🌱このノートブックは、MiniMax H3（マルチモーダル動画AI）をComfyUI上で試すために制作しました。
### 🌱Google Driveへの自動保存に対応しています。（Civitai経由の追加モデル導入は任意・詳細は下のセルのフォームで）
### ‐ ComfyUIをGoogle Colabで起動し、MiniMax H3によるテキスト・画像・動画・音声を組み合わせた動画生成を体験できます。
#### 💡GoogleColab の有料プラン（**コンピューティングユニット節約のためL4 24GB推奨。フル性能が必要な場合はA100 80GBも可**）で使用できるランタイム（GPU）が必要になります。
#### 💾このNotebookは、H3のモデル本体（合計約40〜60GB）をGoogle Driveに保存して使い回す方式です。**初回だけダウンロードが発生し、2回目以降はDriveのファイルをそのまま再利用するので起動が速くなります。**
#### ⚠️H3は量子化版（ComfyUI向けpruned INT8構成）でもR2V一式で約40GB前後のモデル容量が必要です。T4等の非力なGPUでは動作しない可能性が高いため、ランタイムはA100かL4を選択してください。
### 🌱各種生成ツールのコツなど詳しい使い方については、私のYoutbeチャンネルの中で解説しています。
### 📺『ＡＩみちくさｃｈ』 (https://www.youtube.com/channel/UC84fyKjiilxssZVxhE_RiaA)
### 🍀制作者：ざすこ（道草 雑草子） Xアカウント：https://x.com/zasuko_michiksa
##### 🥰このノートブックが役に立ったよ～！と思った方は、感想やご要望などコメント頂けますと幸いです。
---
## 🔗 事前準備：Pinggy Token の設定
このNotebookは、Google Colab上のComfyUIを外部（ブラウザ・PC側ツール等）から使えるように、**Pinggy**というトンネルサービスで一時的な公開URLを発行します。ComfyUIを開くには、この設定が必須です。

> 💡 **Pinggyとは**：Colabのサーバーは通常インターネットから直接アクセスできないため、Pinggyが「Colab内で動いているComfyUI」と「あなたのブラウザ」をつなぐ一時的な公開URL（トンネル）を作ってくれます。このURLは**このColabセッション限りの一時URL**で、Notebookを再起動すると変わります。

**① Pinggyアカウント・トークンの取得**
1. [https://pinggy.io](https://pinggy.io) でアカウントを作成（無料枠あり、Proにすると60分の接続制限が外れます）
2. ダッシュボードから接続トークン（アクセストークン）を取得してコピー

**② Google Colab シークレットへの登録**
1. 左サイドバーの 🔑 **シークレット**（鍵アイコン）を開く
2. **＋ 新しいシークレットを追加** をクリック
3. 名前に `PINGGY_TOKEN`、値にコピーしたトークンを貼り付けて保存
4. **ノートブックからのアクセス** をオンにする

> ⚠️ トークンはシークレットで管理し、コードに直接貼り付けないでください。

---


---

## 🌱 このNotebookは **FastH3版** です

通常版（20ステップ）の Notebook を、**FastH3（4ステップ蒸留＋VSA）** 用に改造したものです。

| | 通常版 | **このFastH3版** |
| --- | --- | --- |
| ステップ数 | 20 | **4** |
| 速度の目安（RTX 4090実測） | — | **1080p×8秒が約3〜4分** |
| 参照画像でキャラを似せる | できる | **ほとんど効きません**（I2Vを使ってください） |

### ⚠️ 実行する前に必ず確認してください

1. **GPUは L4 か A100 を選んでください。**
   高速化のカーネルが Compute Capability 8.0 以上を要求します。
   **無料枠で割り当てられやすい T4（7.5）では効きません。**

2. **これは実験段階の構成です。**
   ComfyUI本体のVSA対応（PR #15958）は 2026-09-01 時点で **未マージ**です。
   動作保証はありません。上流の更新で突然動かなくなることがあります。

3. **解像度 × 秒数の上限があります。**
   `メガピクセル × 秒 ≦ 約16`（VRAM 24GBの場合）。
   768pなら15秒、1080pなら8秒までが目安です。

4. **キャラクターを似せたいなら I2V を使ってください。**
   4ステップでは参照画像からの再現（R2V）がほとんど働きません。

詳しい解説とライセンスの注意は、配布リポジトリの README を読んでください。
<https://github.com/zasuko/zasuko-fasth3-colab>


### 💾 このNotebookは **Drive常駐版** です

モデルを Google ドライブに保存し、2回目以降はダウンロードを省略します。

> **Google ドライブに 44GB 以上の空きが必要です。**
> 無料枠は15GBなので足りません。Google One 100GB 以上のプランが必要です。
> 容量が足りない場合は **Driveレス版**（毎回ダウンロードする版）を使ってください。


# 🥚MiniMax H3用 各種モデルのDL ＆ ComfyUIの起動🐣
##＜はじめる前の準備＞※重要※
### ❶生成に必要な各種モデルファイルのDL用URLを格納する各フォルダの記入欄に記載して下さい。
└🌐GoogleColab上の 📂ComfyUI / models  以下に、生成に必要な各種ファイルをDLします。（DL容量が少ない程起動が速くなります。）
### ❷（▶）ボタンをクリックしてComfyUIを起動

### ❸MiniMax H3はComfyUI公式が v0.30.0 以降で対応。起動セルはComfyUI取得後に自動でバージョン確認・最新化を行います。
### ❹R2V（参照画像から動画）を使う場合は、Workflowsパネルから `MiniMax_H3/video_minimax_h3_r2v` を読み込んでください（自動配置済み）。T2V/I2Vは同フォルダの `video_minimax_h3_t2v` / `video_minimax_h3_i2v` を使用します。


In [ ]:
import base64
# @markdown # 👈この（▶）ボタンをクリックしてComfyUIを起動（☕起動まで約10数分）
# @markdown ### ※停止と再起動もこのボタンで行います。
# @markdown ## 🌱Pinggy接続後、実行ログに表示される青い「ComfyUIを開く」ボタンをクリックしてください。

# ==========================================
# 🔧 PyTorch系ライブラリの軽量チェック（再起動ループ回避版）
# ==========================================
# 重要：このセル内では、現在のColabカーネルに torch / triton を直接 import しません。
# 理由：torchをimportすると内部でtritonも読み込まれ、その後pipがtriton周辺を確認しただけで
#       Colabが「セッションを再起動してください」と表示しやすくなるためです。
# 方針：別プロセスで確認し、必要な場合だけ修復します。
import sys, subprocess

CORE_TORCH_PACKAGES = [
    "torch==2.9.0",
    "torchvision==0.24.0",
    "torchaudio==2.9.0",
]


def run(cmd):
    print("$", " ".join(cmd))
    subprocess.check_call(cmd)


def torch_stack_ok():
    check_code = r'''
import sys
try:
    import torch, torchvision, torchaudio
    print("torch:", torch.__version__)
    print("torchvision:", torchvision.__version__)
    print("torchaudio:", torchaudio.__version__)
    print("torch cuda:", torch.version.cuda)
    print("cuda available:", torch.cuda.is_available())
    ok = ("+cu130" in torch.__version__) and ("+cu130" in torchvision.__version__) and ("+cu130" in torchaudio.__version__)
    sys.exit(0 if ok else 2)
except Exception as e:
    print("PyTorch check failed:", repr(e))
    sys.exit(1)
'''
    result = subprocess.run([sys.executable, "-c", check_code], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)
    return result.returncode == 0

if not torch_stack_ok():
    print("=" * 70)
    print("🔧 torch / torchvision / torchaudio を CUDA 13.0版(cu130)にそろえます")
    print("※必要な時だけ修復します。修復後に「セッションを再起動してください」と表示された場合は、")
    print("　その時だけ再起動し、このセルより上には戻らずこのセルから再実行してください。")
    print("=" * 70)
    run([sys.executable, "-m", "pip", "install", "--no-cache-dir", *CORE_TORCH_PACKAGES,
         "--index-url", "https://download.pytorch.org/whl/cu130"])
    if not torch_stack_ok():
        raise RuntimeError("torch / torchvision / torchaudio の修復に失敗しました。ランタイムを新規にして再実行してください。")

print("✅ PyTorch系ライブラリ確認OK。このままComfyUI起動処理に進みます。")

# ==========================================
# 🧪 2026-08-07追記: cu130実験版について
# ==========================================
# 実測で判明した点: A100(SM80)はSol-AttnのTritonカーネルが非対応で"dense fallback"
# (高速化なしの通常Attention)になり、Sol-Attnの速度メリットが出ない
# (詳細はGenekoナレッジ minimax-h3.md 参照)。一方、comfy_kitchenの高速CUDAバックエンドは
# A100を含むSM80以上をサポートしているが、cu128のままだと無効化(disabled:True)されたまま
# だった。この実験版では torch を cu130 に切り替え、comfy_kitchen側だけでも高速化される
# か検証する。Sol-Attn自体のA100非対応は解消されない見込み(未検証)。
# Colabで「セッションを再起動してください」と出た場合は、その時だけ再起動し、
# このセル(GPUチェックより上のtorchチェックセル)から再実行すること。

# ==========================================
# 🎮 GPUチェック（A100 または L4 を確認、それ以外は停止）
# ==========================================
# @markdown ### GPUチェック
# @markdown 🌱このNotebookはモデルをGoogle Driveに保存して使い回す方式なので、
# @markdown **コンピューティングユニット節約のためL4 24GB（R2Vの短尺～中解像度向け）を推奨**します。フル性能が必要な場合はA100 80GBも使えます。
# @markdown それ以外のGPU（T4等）はVRAM的に厳しいため、チェックを外さない限り止めます。
require_a100_or_l4 = True  # @param {type:"boolean"}
# 🧪 2026-08-07注記: 実測でA100(SM80)はSol-Attnの高速化が効かない(denseフォールバック)ことが
# 判明しました。L4は標準構成で動作確認済みです。A100だから速いとは限らない点に注意してください。


def _check_gpu_for_h3(require_a100_or_l4):
    import subprocess
    try:
        gpu_name = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
        ).strip().splitlines()[0]
    except Exception as e:
        print(f"⚠️ GPU情報を取得できませんでした: {e}")
        return None
    print(f"🎮 検出されたGPU: {gpu_name}")
    if "L4" in gpu_name:
        print("✅ L4を確認しました。コンピューティングユニット節約になるおすすめの構成です。R2Vの短尺・低～中解像度生成なら十分使えます。")
        return "L4"
    if "A100" in gpu_name:
        print("✅ A100を確認しました。フル機能・快適な速度で使えます（L4よりコンピューティングユニットの消費が大きい点に注意）。")
        return "A100"
    print("⚠️ A100/L4ではないGPUです。MiniMax H3（量子化モデルでも約40GB前後）はVRAM不足で失敗する可能性が高いです。")
    if require_a100_or_l4:
        raise RuntimeError(
            "MiniMax H3にはL4 24GB（推奨・コンピューティングユニット節約）またはA100 80GBランタイムが必要です。\n"
            "『ランタイム』→『ランタイムのタイプを変更』でL4かA100を選択し、\n"
            "『ランタイムを接続解除して削除』後にこのセルを再実行してください。\n"
            "（このチェックを飛ばしたい場合は require_a100_or_l4 のチェックを外してください）"
        )
    print("➡️ require_a100_or_l4がオフのため、このまま続行します（自己責任）。")
    return gpu_name

_detected_gpu_for_h3 = _check_gpu_for_h3(require_a100_or_l4)
using_L4_GPU = bool(_detected_gpu_for_h3) and "L4" in str(_detected_gpu_for_h3)
include_manager = True # @param {type:"boolean"}

# ==========================================
# 📁 Google Drive マウント
# ==========================================
# @markdown ## 📁 Google Driveに接続
# @markdown ComfyUIの出力ファイルをバックアップする場合はチェックを入れてください

# @markdown ---
# @markdown ### Google Driveを使用する
use_google_drive = True  # @param {type:"boolean"}

# @markdown ---

if use_google_drive:
    from google.colab import drive
    _drive_mount_ok = False
    for _mount_attempt in range(3):
        try:
            drive.mount('/content/drive', force_remount=(_mount_attempt > 0))
            _drive_mount_ok = True
            break
        except Exception as _mount_err:
            print(f"⚠️ Google Driveのマウントに失敗しました（{_mount_attempt + 1}/3回目）: {_mount_err}")
            print("   Googleの認証ポップアップが出ていたら、許可まで進めてから少し待ってください。")
            import time as _mount_time
            _mount_time.sleep(5)
    if not _drive_mount_ok:
        raise RuntimeError(
            "Google Driveのマウントに3回失敗しました。\n"
            "『ランタイム』→『ランタイムを接続解除して削除』後にもう一度実行するか、\n"
            "ブラウザのポップアップブロック・サードパーティCookie設定を確認してください。"
        )

    print("=" * 70)
    print("✅ Google Driveがマウントされました")
    print("📁 マウント先: /content/drive/MyDrive")
    print("=" * 70)
else:
    print("=" * 70)
    print("⏸️  Google Driveは使用しません")
    print("💡 出力ファイルは/content/ComfyUI/outputに保存されます")
    print("   （ランタイム終了時に消えるので注意してください）")
    print("=" * 70)


# ==========================================
# 📤 Google Drive 出力設定
# ==========================================
# @markdown ## 📤 Google Drive 出力設定
# @markdown ### ComfyUIの出力先をGoogle Driveに変更する
enable_gdrive_output = True  # @param {type:"boolean"}

# @markdown ### 出力先フォルダ設定
GDRIVE_OUTPUT = "/content/drive/MyDrive/ComfyUI_output"  # @param {type:"string"}

# @markdown ---

# 出力フォルダの作成
if use_google_drive and enable_gdrive_output:
    from pathlib import Path
    output_path = Path(GDRIVE_OUTPUT)

    if output_path.exists():
        print(f"📁 既存のフォルダを使用: {GDRIVE_OUTPUT}")
    else:
        output_path.mkdir(parents=True, exist_ok=True)
        print(f"📁 新規フォルダを作成: {GDRIVE_OUTPUT}")

    print("=" * 70)
    print("✅ ComfyUIの出力先がGoogle Driveに設定されます")
    print(f"📁 保存先: {GDRIVE_OUTPUT}")
    print("💡 生成したファイルが直接Google Driveに保存されます")
    print("=" * 70)
elif enable_gdrive_output and not use_google_drive:
    print("=" * 70)
    print("⚠️ Google Drive出力が有効ですが、Google Driveがマウントされていません")
    print("💡 use_google_driveにチェックを入れてください")
    print("=" * 70)
else:
    print("=" * 70)
    print("⏸️  Google Drive出力は無効です")
    print("💡 出力ファイルは/content/ComfyUI/outputに保存されます")
    print("=" * 70)


%cd /content
from IPython.display import clear_output
clear_output()
# torch / triton はColab標準または上のチェック結果を使います。
# xformersだけを --no-deps で入れ、torch/tritonを勝手に入れ替えないようにします。
!python -m pip install -q torchsde einops diffusers accelerate
# cu130 experiment: xformers 0.0.32.post1 is built for cu128, compatibility with cu130 unconfirmed.
# Skipping to avoid conflicts (ComfyUI runs fine without xformers).
# !python -m pip install -q --no-deps xformers==0.0.32.post1
# pipがtorch/triton系を別バージョンへ動かさないための制約ファイル
# Colabの通常Pythonセルでは bash の here-document が壊れやすいため、Pythonで安全に書き出します。
with open('/content/torch_core_constraints.txt', 'w', encoding='utf-8') as f:
    f.write('''torch==2.9.0
torchvision==0.24.0
torchaudio==2.9.0
triton==3.5.0
numpy==2.0.2
''')
!pip install av spandrel albumentations onnx opencv-python onnxruntime -c /content/torch_core_constraints.txt
!pip install color-matcher -c /content/torch_core_constraints.txt
!pip install "protobuf==5.29.1" -c /content/torch_core_constraints.txt
!pip install onnxruntime-gpu -c /content/torch_core_constraints.txt
# 🌱 FastH3版: ComfyUI公式ではなく、VSA対応の作業ブランチを使う。
# 本家PR #15958 (Minimax-H3: support FastVideo VSA) は 2026-09-01 時点で未マージのため、
# kijai氏の vsa ブランチを、動作確認済みのコミットに固定して取得する。
FASTH3_COMFYUI_COMMIT = "10febb01d7be73d1491cf5e5347b5ab8b6c2c09e"  # @param {type:"string"}
!git clone --branch vsa --single-branch https://github.com/kijai/ComfyUI.git /content/ComfyUI
!git -C /content/ComfyUI switch -c fasth3-vsa-fixed {FASTH3_COMFYUI_COMMIT}
print("📌 ComfyUI(vsa) を固定コミットで取得しました:", FASTH3_COMFYUI_COMMIT)

# 🌱 FastH3版: 固定コミットを取得しているので git pull はしない（更新すると壊れる）
%cd /content/ComfyUI
def _check_comfyui_for_fasth3():
    import os
    head = os.popen("git -C /content/ComfyUI rev-parse HEAD").read().strip()
    print("ComfyUI HEAD:", head)
    if head.startswith("10febb01"):
        print("✅ 動作確認済みのコミットです。")
    else:
        print("⚠️ 想定と違うコミットです。上流が変わった可能性があります。")
    ok = os.path.exists("/content/ComfyUI/comfy_extras/nodes_minimax_h3.py")
    print("MiniMax H3ノード:", "✅ あり" if ok else "❌ 見つかりません")
_check_comfyui_for_fasth3()
%cd /content
!pip install -r /content/ComfyUI/requirements.txt -c /content/torch_core_constraints.txt
clear_output()

%cd /content/ComfyUI/custom_nodes
# 🌱 MiniMax H3の公式ワークフローはComfyUIコアだけで完結し、カスタムノードは不要と確認済み
# （ComfyMathExpression含む全ノードがcomfy_extras=コア機能）。起動を速くするため、
# 以前入れていた汎用カスタムノード群（GGUF/VideoHelperSuite/KJNodes/rgthree/essentials/
# LogicUtils/VFI/Frame-Interpolation/PainterI2V）は撤去し、ComfyUI-Managerだけ任意で残す。
if include_manager:
    !git clone https://github.com/ltdrdata/ComfyUI-Manager

# 🌱 FastH3版: 従来のSaganaki22版Sol-Attn（ノード名 SolAttentionPatch）は使わない。
# FastH3のVSAは comfy-kitchen 側のCUDAカーネル(sol_attn)と、専用の暫定ノード
# sol_attn_minimax_v5.py（ノード名 SolAttnMiniMax）の組み合わせで動く。
print("=" * 70)
print("🌱 FastH3用のSol-Attn一式を導入します")
print("=" * 70)

# ① comfy-kitchen の sol_attn 入り wheel
#    PyPIの最新版(0.2.31 / 2026-08-13)にはsol_attnが未収録のため、
#    Apache-2.0の条件に従って再配布しているビルド済みwheelを使う。
import sys as _sys
_ck_py = "cp312-abi3" if _sys.version_info >= (3, 12) else ("cp311-cp311" if _sys.version_info >= (3, 11) else "cp310-cp310")
_ck_wheel = f"comfy_kitchen-0.2.31-{_ck_py}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
_ck_url = "https://github.com/zasuko/zasuko-fasth3-colab/releases/download/comfy-kitchen-solattn/" + _ck_wheel
print(f"⬇️ comfy-kitchen wheel を取得します: {_ck_wheel}")
!wget -q --show-progress -O /content/{_ck_wheel} "{_ck_url}"
!pip install -q --force-reinstall --no-deps /content/{_ck_wheel}

def _check_sol_attn():
    import importlib
    try:
        ck = importlib.import_module("comfy_kitchen")
        importlib.reload(ck)
        has = hasattr(ck, "sol_attn")
        print("comfy_kitchen sol_attn:", "✅ True" if has else "❌ False")
        if not has:
            print("   ⚠️ sol_attnが入っていません。PyPI版が優先された可能性があります。")
        return has
    except Exception as e:
        print("❌ comfy_kitchen の読み込みに失敗:", repr(e))
        return False
_check_sol_attn()

# ② 暫定カスタムノード sol_attn_minimax_v5.py
#    comfy-kitchen PR #117 に添付されているファイルを直接取得する（再配布はしない）。
_solattn_node_url = "https://github.com/user-attachments/files/31576773/sol_attn_minimax_v5.py"
!wget -q -O /content/ComfyUI/custom_nodes/sol_attn_minimax_v5.py "{_solattn_node_url}"
import os as _os_chk
if _os_chk.path.exists("/content/ComfyUI/custom_nodes/sol_attn_minimax_v5.py"):
    print("✅ sol_attn_minimax_v5.py を配置しました")
else:
    print("❌ sol_attn_minimax_v5.py を取得できませんでした。上流のURLが変わった可能性があります。")

# ③ GPUの確認（Sol-Attnのカーネルは Compute Capability 8.0 以上を要求する）
def _check_gpu_for_fasth3():
    import subprocess
    try:
        name = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
            text=True).strip()
        print("GPU:", name)
        cc = float(name.split(",")[1])
        if cc < 8.0:
            print("=" * 70)
            print("❌ このGPUではFastH3の高速化が効きません（Compute Capability 8.0未満）")
            print("   T4(7.5)は対象外です。ランタイムのタイプを L4 または A100 に変更してください。")
            print("=" * 70)
        else:
            print("✅ Sol-Attn対応のGPUです（Compute Capability", cc, "）")
    except Exception as e:
        print("⚠️ GPUを確認できませんでした:", repr(e))
_check_gpu_for_fasth3()

if include_manager:
    %cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
    !pip install -r requirements.txt -c /content/torch_core_constraints.txt

# 🆕 protobufの最終調整
print("=" * 70)
print("🔧 Fixing protobuf version conflicts...")
!pip install protobuf==5.29.1 -c /content/torch_core_constraints.txt
print("✅ Protobuf version fixed")
print("=" * 70)

clear_output()

%cd /content/ComfyUI

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import subprocess
import sys
from pathlib import Path

def install_apt_packages():
    packages = ['aria2']

    try:
        # Run apt install silently (using -qq)
        subprocess.run(
            ['apt-get', '-y', 'install', '-qq'] + packages,
            check=True,
            capture_output=True
        )
        print("✓ apt packages installed")
    except subprocess.CalledProcessError as e:
        print(f"✗ Error installing apt packages: {e.stderr.decode().strip() or 'Unknown error'}")

print("Installing apt packages...")
install_apt_packages()

def download_with_aria2c(link, folder="/content/ComfyUI/models/loras", hf_token=None):
    import os
    from urllib.parse import urlparse, unquote

    # Hugging Faceなど、URLの末尾に正式なファイル名が入っている通常リンク用
    path_name = os.path.basename(urlparse(link).path)
    filename = unquote(path_name) if path_name else "downloaded_model.safetensors"
    header_arg = ""
    if hf_token and "huggingface.co" in link.lower():
        header_arg = f' --header="Authorization: Bearer {hf_token}"'
    command = f"aria2c --console-log-level=error -c -x 16 -s 16 -k 1M{header_arg} \"{link}\" -d \"{folder}\" -o \"{filename}\""

    print("Executing download command:")
    print(command.replace(hf_token, "***") if hf_token else command)

    os.makedirs(folder, exist_ok=True)
    get_ipython().system(command)

    return filename

def add_civitai_token_to_url(civitai_link, civitai_token=None):
    """CivitaiのURLにtokenを安全に追加する。既にtokenがある場合はそのまま使う。"""
    from urllib.parse import urlparse, parse_qsl, urlencode, urlunparse

    parsed = urlparse(civitai_link)
    if not parsed.scheme or not parsed.netloc:
        raise ValueError("Invalid Civitai URL format. Please use a link like: https://civitai.com/api/download/models/1523247?...")

    query = dict(parse_qsl(parsed.query, keep_blank_values=True))
    if civitai_token and "token" not in query:
        query["token"] = civitai_token

    return urlunparse((parsed.scheme, parsed.netloc, parsed.path, parsed.params, urlencode(query), parsed.fragment))

def pick_new_downloaded_file(folder, before_files):
    """wget実行後に増えたファイルから、実体のある最新ファイルを推定する。"""
    import os

    after_files = set(os.listdir(folder))
    candidates = []
    for name in after_files - before_files:
        path = os.path.join(folder, name)
        if os.path.isfile(path) and os.path.getsize(path) > 0:
            # wgetの一時ファイルやログっぽいものは除外
            if not name.endswith((".tmp", ".aria2")):
                candidates.append(path)

    if not candidates:
        return None

    candidates.sort(key=lambda p: os.path.getmtime(p), reverse=True)
    return candidates[0]

def download_civitai_model(civitai_link, civitai_token, folder="/content/ComfyUI/models/loras"):
    """
    Civitai専用ダウンロード。
    重要：-Oで固定名を付けず、wgetの --content-disposition / --trust-server-names を使う。
    これにより、Civitaiが返す正式ファイル名（例：FlowCamera_epoch67.safetensors）で保存する。
    """
    import os
    import subprocess

    os.makedirs(folder, exist_ok=True)
    civitai_url = add_civitai_token_to_url(civitai_link, civitai_token)

    before_files = set(os.listdir(folder))

    cmd = [
        "wget",
        "--max-redirect=10",
        "--content-disposition",
        "--trust-server-names",
        "--show-progress",
        "-P", folder,
        civitai_url,
    ]

    print("Downloading from Civitai with official filename...")
    print("$ " + " ".join([f'\"{c}\"' if " " in c or "&" in c or "?" in c else c for c in cmd]))

    result = subprocess.run(cmd, text=True)
    if result.returncode != 0:
        print("❌ Civitai download failed.")
        return False

    downloaded_path = pick_new_downloaded_file(folder, before_files)

    # 同名ファイルが既に存在していて新規差分が取れない場合の保険：フォルダ内の最新safetensorsを表示
    if downloaded_path is None:
        safetensors = [
            os.path.join(folder, f) for f in os.listdir(folder)
            if f.lower().endswith((".safetensors", ".ckpt", ".pt", ".pth", ".bin")) and os.path.isfile(os.path.join(folder, f))
        ]
        if safetensors:
            downloaded_path = max(safetensors, key=os.path.getmtime)

    if downloaded_path and os.path.exists(downloaded_path) and os.path.getsize(downloaded_path) > 0:
        filename = os.path.basename(downloaded_path)
        print(f"✅ Civitai model downloaded successfully: {downloaded_path}")
        print(f"📛 Saved filename: {filename}")
        return filename

    print("❌ Civitai download finished, but the saved file could not be detected.")
    return False

def download_lora(link, folder="/content/ComfyUI/models/loras", civitai_token=None):
    """
    Download a model file, automatically detecting if it's a Civitai link or huggingface download.

    Args:
        link: The download URL (either huggingface or Civitai)
        folder: Destination folder for the download
        civitai_token: Optional token for Civitai downloads (required if link is from Civitai)

    Returns:
        The filename of the downloaded model
    """
    if "civitai.com" in link.lower():
        if not civitai_token:
            print("⚠️ Civitaiトークンが未設定のため、トークンなしでダウンロードを試します。")
            print("   非公開・年齢制限・ログイン必須モデルの場合は失敗することがあります。")
        return download_civitai_model(link, civitai_token, folder)
    else:
        return download_with_aria2c(link, folder)

def model_download(url: str, dest_dir: str, filename: str = None, silent: bool = True, hf_token: str = None) -> bool:
    """
    Colab-optimized download with aria2c

    Args:
        url: Download URL
        dest_dir: Target directory (will be created if needed)
        filename: Optional output filename (defaults to URL filename)
        silent: If True, suppresses all output (except errors)
        hf_token: Optional Hugging Face token. Added as an Authorization header
            only when the URL host is huggingface.co (rate-limit avoidance).

    Returns:
        bool: True if successful, False if failed
    """
    try:
        # Create destination directory
        Path(dest_dir).mkdir(parents=True, exist_ok=True)

        # Set filename if not specified
        if filename is None:
            filename = url.split('/')[-1].split('?')[0]  # Remove URL parameters

        # Build command
        cmd = [
            'aria2c',
            '--console-log-level=error',
            '-c', '-x', '16', '-s', '16', '-k', '1M',
            '-d', dest_dir,
            '-o', filename,
            url
        ]
        if hf_token and "huggingface.co" in url.lower():
            cmd.insert(1, f'--header=Authorization: Bearer {hf_token}')

        # Add silent flags if requested
        if silent:
            cmd.extend(['--summary-interval=0', '--quiet'])
            print(f"Downloading {filename}...", end=' ', flush=True)

        # Run download
        result = subprocess.run(cmd, check=True, capture_output=True, text=True)

        if silent:
            print("Done!")
        else:
            print(f"Downloaded {filename} to {dest_dir}")
        return filename

    except subprocess.CalledProcessError as e:
        error = e.stderr.strip() or "Unknown error"
        print(f"\nError downloading {filename}: {error}")
        return False
    except Exception as e:
        print(f"\nError: {str(e)}")
        return False

# ========================================
# ダウンロード処理関数
# ========================================

def parse_urls(url_string):
    """
    複数行のURL文字列をパースして、有効なURLのリストを返す

    Args:
        url_string: 改行区切りのURL文字列

    Returns:
        list: 有効なURLのリスト
    """
    urls = []
    for line in url_string.strip().split('\n'):
        line = line.strip()
        # コメント行（#始まり）と空行をスキップ
        if line and not line.startswith('#') and line.startswith('http'):
            urls.append(line)
    return urls

# ========================================
# 🌱💾 Google Driveへのモデル永続保存
# ========================================
# H3のモデル本体（合計約40〜60GB）を、毎回Colabへダウンロードし直すのではなく
# Google Driveの下記フォルダに保存し、2回目以降はそれを再利用する。
# 初回だけダウンロードが発生し、2回目以降はセットアップがぐっと速くなる。

# @markdown ---
# @markdown ## 💾 Google Driveへのモデル永続保存
# @markdown モデル本体をGoogle Driveの以下のフォルダに保存し、次回以降は再ダウンロードをスキップします。
DRIVE_MODEL_ROOT = "/content/drive/MyDrive/MiniMaxH3_Models"  # @param {type:"string"}
# @markdown 容量が気になる方はフォルダパスを変更してください（フォルダは自動作成されます）。
# @markdown チェックを外すと、Driveに保存せず毎回ローカルへダウンロードする従来動作になります（Colab終了で消えます）。
persist_models_to_drive = True  # @param {type:"boolean"}

# @markdown ---
# @markdown ## 💾 モデル読み込み方式（既定: ステージングなし・L4実測で安定）
# @markdown Driveのモデルファイルへ直接アクセスします（ローカルへのコピー待ち時間なし）。基本的にONのままで問題ありません。チェックを外すと、従来どおり一度ローカルディスクへコピーしてから使います。
skip_local_staging = True  # @param {type:"boolean"}

import os
import shutil
from pathlib import Path

# 既知のH3公式ファイルは、想定サイズ(GiB)を下回っていたら壊れているとみなして再ダウンロードする
MODEL_MIN_GIB_HINTS = {
    "minimax_h3_ref2va_pruned_int8_convrot.safetensors": 19.0,
    "minimax_h3_fl2va_pruned_int8_convrot.safetensors": 19.0,
    "minimax_h3_ref2va_int8_convrot.safetensors": 19.0,
    "minimax_h3_fl2va_int8_convrot.safetensors": 19.0,
    "minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors": 21.0,
    "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors": 14.0,
    "qwen3vl_32b_minimax_h3_int8_convrot.safetensors": 30.0,
    "minimax_h3_video_vae_fp16.safetensors": 4.5,
    "minimax_h3_audio_vae_fp32.safetensors": 0.5,
}


def ensure_model_persistent(url, comfy_folder, filename=None, min_expected_gib=None, civitai_token=None, hf_token=None):
    """
    Google Driveに保存したモデルを再利用しつつ、必要な時だけダウンロードする。
    さらに、実行時の読み込み速度のためColabのローカルディスクへステージング（コピー）してから使う。

    1. Drive上に既に十分なサイズのファイルがあれば、再ダウンロードしない。
       無ければ、まずColabのローカルディスクへダウンロードし、サイズを確認してからDriveへ移動する。
    2. ComfyUIのmodelsフォルダへは、Driveのファイルを直接シンボリックリンクするのではなく、
       いったんColabのローカルディスク（/content/h3_model_stage/）へコピーしてから、
       そのローカルコピーへシンボリックリンクを張る。
       Google Drive越しの読み込み（FUSEマウント）はネットワーク経由になり遅くなりがちなので、
       実際にComfyUIが読みに行く先はローカルSSD相当にすることで高速化する。
       ローカルの空き容量が足りない場合は、従来通りDriveへ直接シンボリックリンクする。
    """
    from urllib.parse import urlparse, unquote

    comfy_dest_dir = Path(comfy_folder)
    comfy_dest_dir.mkdir(parents=True, exist_ok=True)

    is_civitai = "civitai.com" in url.lower()

    if not filename and not is_civitai:
        path_name = os.path.basename(urlparse(url).path)
        filename = unquote(path_name) if path_name else "model.safetensors"

    if not persist_models_to_drive or is_civitai:
        # Drive保存なし（従来通りローカルへ直接ダウンロード）。
        # CivitaiはダウンロードするまでファイルGname確定しないため、Drive方式の対象外にしている。
        if is_civitai:
            download_lora(url, folder=str(comfy_dest_dir), civitai_token=civitai_token)
        else:
            model_download(url, str(comfy_dest_dir), filename=filename)
        return

    if min_expected_gib is None:
        min_expected_gib = MODEL_MIN_GIB_HINTS.get(filename)

    drive_dir = Path(DRIVE_MODEL_ROOT)
    drive_dir.mkdir(parents=True, exist_ok=True)
    drive_path = drive_dir / filename
    comfy_path = comfy_dest_dir / filename

    min_expected_bytes = int(min_expected_gib * (1024 ** 3) * 0.98) if min_expected_gib else 1

    if drive_path.is_file() and drive_path.stat().st_size >= min_expected_bytes:
        print(f"✅ Driveに保存済みのモデルを再利用: {filename}（{drive_path.stat().st_size / 1024**3:.2f} GiB）")
    else:
        print(f"⬇️ Driveに未保存のため新規ダウンロード: {filename}")
        tmp_dir = Path("/content/h3_model_tmp")
        tmp_dir.mkdir(parents=True, exist_ok=True)
        import time as _dl_time_mod
        _dl_started = _dl_time_mod.time()
        tmp_filename = model_download(url, str(tmp_dir), filename=filename, hf_token=hf_token)
        _dl_elapsed = _dl_time_mod.time() - _dl_started
        tmp_path = tmp_dir / (tmp_filename or filename)
        if not tmp_path.is_file():
            print(f"❌ ダウンロードに失敗しました: {filename}")
            return
        if min_expected_gib and tmp_path.stat().st_size < min_expected_bytes:
            print(f"⚠️ ファイルサイズが想定より小さいです（{tmp_path.stat().st_size / 1024**3:.2f} GiB）。破損の可能性があるため保存を中止します。")
            return
        _dl_mbps = (tmp_path.stat().st_size / 1024 / 1024) / _dl_elapsed if _dl_elapsed > 0 else 0
        print(f"⬇️ ダウンロード完了: {filename}（{_dl_elapsed:.1f}秒、{_dl_mbps:.1f} MB/s、aria2c 16並列）")
        print(f"📦 Google Driveへ移動中...（{tmp_path.stat().st_size / 1024**3:.2f} GiB、少し時間がかかります）")
        if drive_path.exists():
            drive_path.unlink()
        shutil.move(str(tmp_path), str(drive_path))
        print(f"✅ Driveへ保存しました: {drive_path}")

    # ── ここからローカルステージング ──
    stage_dir = Path("/content/h3_model_stage")
    stage_dir.mkdir(parents=True, exist_ok=True)
    stage_path = stage_dir / filename
    drive_size = drive_path.stat().st_size

    link_target = drive_path  # 既定はDriveへ直接（空き容量不足などの保険）

    stage_ok = stage_path.is_file() and not stage_path.is_symlink() and stage_path.stat().st_size == drive_size
    if skip_local_staging:
        print(f"🧪 実験: ローカルステージングをスキップし、Driveへ直接シンボリックリンクします: {filename}")
        link_target = drive_path
    elif stage_ok:
        print(f"✅ ローカルステージング済みを再利用: {filename}")
        link_target = stage_path
    else:
        free_gib = shutil.disk_usage("/content").free / (1024 ** 3)
        needed_gib = drive_size / (1024 ** 3) + 5  # 5GiBの余裕を見る
        if free_gib < needed_gib:
            print(f"⚠️ ローカルディスクの空きが足りないため（空き{free_gib:.1f}GiB／必要{needed_gib:.1f}GiB）、Driveから直接読み込みます: {filename}")
        else:
            print(f"📥 ローカルディスクへステージング中...（{drive_size / 1024**3:.2f} GiB、少し時間がかかります）")
            stage_tmp = stage_path.with_suffix(stage_path.suffix + ".copying")
            import time as _stage_time_mod
            _stage_started = _stage_time_mod.time()
            with open(drive_path, "rb") as _src, open(stage_tmp, "wb") as _dst:
                shutil.copyfileobj(_src, _dst, length=64 * 1024 * 1024)
            _stage_elapsed = _stage_time_mod.time() - _stage_started
            _stage_mbps = (drive_size / 1024 / 1024) / _stage_elapsed if _stage_elapsed > 0 else 0
            stage_tmp.replace(stage_path)
            print(f"✅ ローカルステージング完了: {filename}（{_stage_elapsed:.1f}秒、{_stage_mbps:.1f} MB/s）")
            link_target = stage_path

    if comfy_path.is_symlink():
        if comfy_path.resolve() != link_target.resolve():
            comfy_path.unlink()
            comfy_path.symlink_to(link_target)
    elif comfy_path.exists():
        print(f"⚠️ {comfy_path} は既にシンボリックリンク以外のファイルとして存在するため、そのままにします。")
    else:
        comfy_path.symlink_to(link_target)


def download_models_to_folder(url_string, folder_path, civitai_token=None):
    """
    指定されたフォルダに複数のモデルをダウンロード（Drive永続化対応版）。

    Args:
        url_string: 改行区切りのURL文字列
        folder_path: ダウンロード先フォルダパス
        civitai_token: Civitai APIトークン（オプション）
    """
    urls = parse_urls(url_string)

    if not urls:
        return

    print(f"\n📂 対象フォルダ: {folder_path}")
    print(f"   ファイル数: {len(urls)}")

    for i, url in enumerate(urls, 1):
        if "civitai.com" in url.lower():
            display_name = "Civitai公式ファイル名（ダウンロード後に確定）"
            hint_key = None
        else:
            display_name = url.split('/')[-1].split('?')[0]
            hint_key = display_name
        print(f"   [{i}/{len(urls)}] {display_name}")
        ensure_model_persistent(
            url,
            folder_path,
            filename=None if "civitai.com" in url.lower() else display_name,
            min_expected_gib=MODEL_MIN_GIB_HINTS.get(hint_key) if hint_key else None,
            civitai_token=civitai_token,
        )

    print(f"✓ 完了: {folder_path}\n")


# ========================================
# 📥 各種モデルのダウンロード設定
# ========================================

# @markdown # 📥 モデルダウンロード設定
# @markdown 各フォルダごとに、使用したいモデルのダウンロードURLを入力してください。
# @markdown 複数のURLは改行で区切って入力できます。

# @markdown ---
# @markdown ## 🔑 Civitai API Token（任意・H3本体では不要）
# @markdown H3のモデル本体はHugging Faceから取得するため、この設定はH3を使うだけなら不要です。Civitai経由で追加のカスタムモデルをDLしたい場合のみ、`CIVITAI_KEY`という名前でColabの🔑シークレットに登録してください。

civitai_token = ""

try:
    from google.colab import userdata
    civitai_token = userdata.get('CIVITAI_KEY')
    if civitai_token:
        civitai_token = civitai_token.strip()
        print("✅ Civitai API キーをシークレットから読み込みました")
except Exception:
    pass

if not civitai_token:
    print("ℹ️ Civitai APIキー未設定です（H3の利用には不要なので、そのままで問題ありません）")

# @markdown ---
# @markdown ## 🔑 Hugging Face API Token（任意・推奨）
# @markdown H3のモデル本体はHugging Faceの公開リポジトリ（`Comfy-Org/MiniMax-H3`）にあり、トークンが無くてもダウンロードできます。
# @markdown ただし、トークンを登録しておくとダウンロード時のレート制限（アクセス制限）にかかりにくくなるため、設定を推奨します。
# @markdown
# @markdown **① トークンの取得**
# @markdown 1. [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) を開く（要ログイン）
# @markdown 2. **＋ Create new token** をクリック（種類は「Read」で十分）
# @markdown 3. 名前を入力して作成 → 表示されたトークンをコピー
# @markdown
# @markdown **② Google Colab シークレットへの登録**
# @markdown 1. 左サイドバーの 🔑 **シークレット**（鍵アイコン）を開く
# @markdown 2. **＋ 新しいシークレットを追加** をクリック
# @markdown 3. 名前に `HF_TOKEN`、値にコピーしたトークンを貼り付けて保存
# @markdown 4. **ノートブックからのアクセス** をオンにする

hf_token = ""

try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        hf_token = hf_token.strip()
        print("✅ Hugging Face トークンをシークレットから正常に読み込みました")
    else:
        print("ℹ️ 'HF_TOKEN' は見つかりましたが、中身が空っぽのようです（公開モデルなのでトークンなしでも続行できます）")
except Exception as e:
    print(f"ℹ️ Hugging Face トークンの読み込みをスキップします: {e}")
    print("   （`Comfy-Org/MiniMax-H3` は公開リポジトリのため、トークンが無くてもダウンロード自体は可能です）")
    hf_token = ""

if not hf_token:
    print("ℹ️ Hugging Faceトークンなしで続行します。レート制限にかかった場合は上記手順でHF_TOKENを設定してください。")

# @markdown ---
# @markdown ## 📁 diffusion_models フォルダ
# @markdown 🌱FastH3（4ステップ蒸留＋VSA）本体・約21.3GiB。これ1つでT2V/I2V/R2Vすべてを兼ねます。※20ステップの通常版とは別物です/I2VならFL2VAが必要。両方入れておけばT2V/I2V/R2Vすべて使える（合計約39GiB）
diffusion_models_url_1 = "https://huggingface.co/Kijai/MiniMax-H3-experimental/resolve/main/minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors" # @param {type:"string"}
diffusion_models_url_2 = "" # @param {type:"string"}
diffusion_models_url_3 = "" # @param {type:"string"}
diffusion_models_url_4 = "" # @param {type:"string"}
diffusion_models_url_5 = "" # @param {type:"string"}
diffusion_models_url_6 = "" # @param {type:"string"}
diffusion_models_url_7 = "" # @param {type:"string"}
diffusion_models_url_8 = "" # @param {type:"string"}

# 入力されたURLを結合
diffusion_models_urls = "\n".join([
    url for url in [
        diffusion_models_url_1,
        diffusion_models_url_2,
        diffusion_models_url_3,
        diffusion_models_url_4,
        diffusion_models_url_5,
        diffusion_models_url_6,
        diffusion_models_url_7,
        diffusion_models_url_8
    ] if url.strip()
])

# @markdown ---
# @markdown ## 📁 text_encoders フォルダ
# @markdown MiniMax H3用 テキストエンコーダー（Qwen3-VL-32B 量子化NVFP4-AWQ版・約14.6GiB）。プロンプトと参照画像・参照動画の理解に使用
text_encoders_url_1 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" # @param {type:"string"}
text_encoders_url_2 = "" # @param {type:"string"}
text_encoders_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
text_encoders_urls = "\n".join([url for url in [text_encoders_url_1, text_encoders_url_2, text_encoders_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 vae フォルダ
# @markdown MiniMax H3用 動画VAEと音声VAE（H3は動画と音声を同時に扱うため両方必須）
vae_url_1 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors" # @param {type:"string"}
vae_url_2 = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors" # @param {type:"string"}
vae_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
vae_urls = "\n".join([url for url in [vae_url_1, vae_url_2, vae_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 clip_vision フォルダ
# @markdown CLIP Visionモデル（画像理解用）
clip_vision_url_1 = "" # @param {type:"string"}
clip_vision_url_2 = "" # @param {type:"string"}
clip_vision_url_3 = "" # @param {type:"string"}

# 入力されたURLを結合
clip_vision_urls = "\n".join([url for url in [clip_vision_url_1, clip_vision_url_2, clip_vision_url_3] if url.strip()])

# @markdown ---
# @markdown ## 📁 loras フォルダ
# @markdown MiniMax H3用の公式高速化LoRA（Lightning/Turbo等）は2026-08-03時点で確認されていないため既定は空欄。必要なLoRAが見つかった場合のみ記入する

loras_url_1 = "" # @param {type:"string"}
loras_url_2 = "" # @param {type:"string"}
loras_url_3 = "" # @param {type:"string"}
loras_url_4 = "" # @param {type:"string"}
loras_url_5 = "" # @param {type:"string"}
loras_url_6 = "" # @param {type:"string"}
loras_url_7 = "" # @param {type:"string"}
loras_url_8 = "" # @param {type:"string"}

# URLをまとめる
loras_urls = "\n".join([url for url in [loras_url_1, loras_url_2, loras_url_3, loras_url_4, loras_url_5, loras_url_6, loras_url_7, loras_url_8] if url])

# @markdown ---
# @markdown ## 📁 audio_encoders フォルダ
# @markdown Audio Encoderモデル（音声付き動画生成用）
audio_encoders_url_1 = "" # @param {type:"string"}
audio_encoders_url_2 = "" # @param {type:"string"}
audio_encoders_url_3 = "" # @param {type:"string"}

# URLをまとめる
audio_encoders_urls = "\n".join([url for url in [audio_encoders_url_1, audio_encoders_url_2, audio_encoders_url_3] if url])

# @markdown ---
# @markdown ## 📁 sam2 フォルダ
# @markdown SAM2セグメンテーションモデル
sam2_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 unet フォルダ
# @markdown UNetモデル（Flux等）
unet_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 checkpoints フォルダ
# @markdown チェックポイントモデル
checkpoints_url_1 = "" # @param {type:"string"}
checkpoints_url_2 = "" # @param {type:"string"}
checkpoints_url_3 = "" # @param {type:"string"}
checkpoints_url_4 = "" # @param {type:"string"}
checkpoints_url_5 = "" # @param {type:"string"}

# 入力されたURLを結合
checkpoints_urls = "\n".join([url for url in [checkpoints_url_1, checkpoints_url_2, checkpoints_url_3, checkpoints_url_4, checkpoints_url_5] if url.strip()])

# @markdown ---
# @markdown ## 📁 controlnet フォルダ
# @markdown ControlNetモデル
controlnet_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 clip フォルダ
# @markdown CLIPモデル
clip_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 upscale_models フォルダ
# @markdown アップスケールモデル
upscale_models_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 embeddings フォルダ
# @markdown Embeddingsモデル
embeddings_urls = "" # @param {type:"string"}

# @markdown ---
# @markdown ## 📁 カスタムフォルダ（自由指定）
# @markdown フォルダパスとURLを指定
custom_folder_path = "" # @param {type:"string"}
custom_folder_urls = "" # @param {type:"string"}


# ========================================
# ダウンロード実行
# ========================================

print("=" * 70)
print("🚀 Starting model downloads...")
print("=" * 70)

# 各フォルダへのダウンロード
download_models_to_folder(diffusion_models_urls, "/content/ComfyUI/models/diffusion_models", civitai_token)
download_models_to_folder(text_encoders_urls, "/content/ComfyUI/models/text_encoders", civitai_token)
download_models_to_folder(vae_urls, "/content/ComfyUI/models/vae", civitai_token)
download_models_to_folder(clip_vision_urls, "/content/ComfyUI/models/clip_vision", civitai_token)
download_models_to_folder(loras_urls, "/content/ComfyUI/models/loras", civitai_token)
download_models_to_folder(audio_encoders_urls, "/content/ComfyUI/models/audio_encoders", civitai_token)
download_models_to_folder(sam2_urls, "/content/ComfyUI/models/sam2", civitai_token)
download_models_to_folder(unet_urls, "/content/ComfyUI/models/unet", civitai_token)
download_models_to_folder(checkpoints_urls, "/content/ComfyUI/models/checkpoints", civitai_token)
download_models_to_folder(controlnet_urls, "/content/ComfyUI/models/controlnet", civitai_token)
download_models_to_folder(clip_urls, "/content/ComfyUI/models/clip", civitai_token)
download_models_to_folder(upscale_models_urls, "/content/ComfyUI/models/upscale_models", civitai_token)
download_models_to_folder(embeddings_urls, "/content/ComfyUI/models/embeddings", civitai_token)

# カスタムフォルダ
if custom_folder_path and custom_folder_urls.strip():
    download_models_to_folder(custom_folder_urls, custom_folder_path, civitai_token)

print("=" * 70)
print("✓ All downloads completed!")
print("=" * 70)

# ========================================
# 🌱 FastH3用ワークフローの配置
# ========================================
# 20ステップ版のワークフローテンプレート生成（Sol-Attn+EasyCache版の自動生成を含む）は、
# FastH3では設定が噛み合わないため丸ごと差し替えています。
#
# 【重要】配布元のサンプル fasth3_vsa_sample.json は「API形式」で、ノードの座標も
# 接続線の情報も持たないため、ComfyUIの画面で開いても真っ白になります。
# ここでは、それを「UI形式」に変換して動作確認済みのものを埋め込んでいます。
# （2026-09-01にローカルのComfyUIで、実際に画面から開けることを目視確認済み）

import os as _os, json as _json, base64 as _b64

h3_workflow_dir = "/content/ComfyUI/user/default/workflows/FastH3"
_os.makedirs(h3_workflow_dir, exist_ok=True)

_WF_T2V_B64 = (
    "eyJpZCI6ImZhc3RoMy10MnYiLCJyZXZpc2lvbiI6MCwibGFzdF9ub2RlX2lkIjoxNiwibGFzdF9saW5rX2lkIjoxOCwibm9kZXMi"
    "Olt7ImlkIjoxLCJ0eXBlIjoiVU5FVExvYWRlciIsInBvcyI6WzAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVy"
    "IjowLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6"
    "WzFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJVTkVUTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbIm1p"
    "bmltYXhfaDNfZmFzdHZpZGVvX3ZzYV9kYXRhZnJlZV8xMzAwc3RlcF80c3RlcF9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLCJk"
    "ZWZhdWx0Il19LHsiaWQiOjIsInR5cGUiOiJNaW5pTWF4SDNTaWdtYVNoaWZ0IiwicG9zIjpbNDYwLDBdLCJzaXplIjpbNDAwLDEw"
    "OF0sImZsYWdzIjp7fSwib3JkZXIiOjcsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6Im1vZGVsIiwidHlwZSI6Ik1PREVMIiwi"
    "bGluayI6MX1dLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6WzJdfV0sInByb3BlcnRp"
    "ZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJNaW5pTWF4SDNTaWdtYVNoaWZ0In0sIndpZGdldHNfdmFsdWVzIjpbMTIuMCwzLjBd"
    "fSx7ImlkIjozLCJ0eXBlIjoiU29sQXR0bk1pbmlNYXgiLCJwb3MiOls5MjAsMF0sInNpemUiOls0MDAsMjEyXSwiZmxhZ3MiOnt9"
    "LCJvcmRlciI6OSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rIjoyfV0sIm91"
    "dHB1dHMiOlt7Im5hbWUiOiJNT0RFTCIsInR5cGUiOiJNT0RFTCIsImxpbmtzIjpbNV19XSwicHJvcGVydGllcyI6eyJOb2RlIG5h"
    "bWUgZm9yIFMmUiI6IlNvbEF0dG5NaW5pTWF4In0sIndpZGdldHNfdmFsdWVzIjpbIlZTQSAoRmFzdFZpZGVvKSIsMC4wLDEuMCww"
    "LCJleGFjdF9rdl9hbmRfcm93cyIsdHJ1ZV19LHsiaWQiOjQsInR5cGUiOiJDTElQTG9hZGVyIiwicG9zIjpbMCwyNjBdLCJzaXpl"
    "IjpbNDAwLDEwOF0sImZsYWdzIjp7fSwib3JkZXIiOjEsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJD"
    "TElQIiwidHlwZSI6IkNMSVAiLCJsaW5rcyI6WzNdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJDTElQTG9h"
    "ZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbInF3ZW4zdmxfMzJiX21pbmltYXhfaDNfbnZmcDRfYXdxLnNhZmV0ZW5zb3JzIiwibWlu"
    "aW1heCIsImRlZmF1bHQiXX0seyJpZCI6NSwidHlwZSI6IlZBRUxvYWRlciIsInBvcyI6WzAsNTIwXSwic2l6ZSI6WzQwMCw2MF0s"
    "ImZsYWdzIjp7fSwib3JkZXIiOjIsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJWQUUiLCJ0eXBlIjoi"
    "VkFFIiwibGlua3MiOls0LDEzXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFFTG9hZGVyIn0sIndpZGdl"
    "dHNfdmFsdWVzIjpbIm1pbmltYXhfaDNfdmlkZW9fdmFlX2ZwMTYuc2FmZXRlbnNvcnMiXX0seyJpZCI6NiwidHlwZSI6IlZBRUxv"
    "YWRlciIsInBvcyI6WzAsNzgwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjMsIm1vZGUiOjAsImlucHV0cyI6"
    "W10sIm91dHB1dHMiOlt7Im5hbWUiOiJWQUUiLCJ0eXBlIjoiVkFFIiwibGlua3MiOlsxNV19XSwicHJvcGVydGllcyI6eyJOb2Rl"
    "IG5hbWUgZm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX2F1ZGlvX3ZhZV9mcDMyLnNh"
    "ZmV0ZW5zb3JzIl19LHsiaWQiOjcsInR5cGUiOiJNaW5pTWF4SDNJbWFnZVRvVmlkZW8iLCJwb3MiOls0NjAsMjYwXSwic2l6ZSI6"
    "WzQwMCwyMzhdLCJmbGFncyI6e30sIm9yZGVyIjo4LCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJjbGlwIiwidHlwZSI6IkNM"
    "SVAiLCJsaW5rIjozfSx7Im5hbWUiOiJ2YWUiLCJ0eXBlIjoiVkFFIiwibGluayI6NH0seyJuYW1lIjoiZmlyc3RfZnJhbWUiLCJ0"
    "eXBlIjoiSU1BR0UiLCJsaW5rIjpudWxsLCJzaGFwZSI6N30seyJuYW1lIjoibGFzdF9mcmFtZSIsInR5cGUiOiJJTUFHRSIsImxp"
    "bmsiOm51bGwsInNoYXBlIjo3fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJwb3NpdGl2ZSIsInR5cGUiOiJDT05ESVRJT05JTkciLCJs"
    "aW5rcyI6WzZdfSx7Im5hbWUiOiJMQVRFTlQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOlsxMV19XSwicHJvcGVydGllcyI6eyJO"
    "b2RlIG5hbWUgZm9yIFMmUiI6Ik1pbmlNYXhIM0ltYWdlVG9WaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJpbnRlZ3JhdGVkX211"
    "bHRpbW9kYWxfZGVzY3JpcHRpb246IFtTaG90IDFdIDJELWFuaW1hdGVkLCBhbmltZSBzdHlsZSwgYSBtZWRpdW0gY2xvc2UtdXAg"
    "c2hvdCBmcmFtZXMgYSBjaGVlcmZ1bCB5b3VuZyB3b21hbiB3aXRoIGRhcmsgaGFpciB0aWVkIHVwIGluIGEgbmVhdCBidW4sIHNp"
    "ZGUgYmFuZ3MgZnJhbWluZyBoZXIgZGVsaWNhdGUgZmFjZSwgYW5kIGRhcmsgcHVycGxlIGV5ZXMgd2VhcmluZyBzbWFsbCBwZWFy"
    "bCBlYXJyaW5ncy4gU2hlIHdlYXJzIGEgbGlnaHQgZ3JheSB0dXJ0bGVuZWNrIGtuaXQgdG9wIHVuZGVyIGEgYmxhY2sgc3BhZ2hl"
    "dHRpLXN0cmFwIG1pbmkgZHJlc3MuIFN0YW5kaW5nIGluIGEgY2xlYW4gbW9kZXJuIHJvb20gd2l0aCB3aGl0ZSB3YWxscyBhbmQg"
    "YSBkYXJrIGRvb3JmcmFtZSwgc2hlIHRpbHRzIGhlciBoZWFkIHNsaWdodGx5IHdpdGggYSBicmlnaHQgc21pbGUsIGxpZ2h0bHkg"
    "cmVzdGluZyBoZXIgcmlnaHQgZmluZ2VydGlwcyBuZWFyIGhlciBjb2xsYXJib25lLiBUaGUgY2FtZXJhIHB1c2hlcyBpbiB3aXRo"
    "IHNtYWxsIGFtcGxpdHVkZSBhdCBzbG93IHNwZWVkIGFzIHRoZSB5b3VuZyB3b21hbiB3aXRoIGEgc3dlZXQsIGNsZWFyIHZvaWNl"
    "IChTMSkgZ2VudGx5IHNheXM6IDxkPltKYXBhbmVzZV0g44GT44KT44Gr44Gh44Gv77yB5LuK5pel44KC5LiA5pel6aCR5by144KN"
    "44GG44Gt44CCPC9kPlxuXG5vdmVyYWxsX3NvdW5kc2NhcGU6IFF1aWV0IGluZG9vciByb29tIHRvbmUsIGEgc29mdCBydXN0bGUg"
    "b2Yga25pdCBmYWJyaWMgYXMgc2hlIG1vdmVzIGhlciBzaG91bGRlciwgYW5kIHRoZSBjbGVhciBzcG9rZW4gdm9pY2UgaW4gYSBu"
    "YXR1cmFsIHJvb20gYWNvdXN0aWMuXG5cbm5vbl9kaWVnZXRpY19tdXNpYzogQSBnZW50bGUsIHVwbGlmdGluZyBhY291c3RpYyBn"
    "dWl0YXIgcGF0dGVybiB3aXRoIHdhcm0gcGlhbm8gbm90ZXMgYXQgYSBtb2RlcmF0ZSB0ZW1wbywgZmFkaW5nIHNvZnRseSBhdCB0"
    "aGUgZW5kLiIsNTQ0LDgzMiwxMjRdfSx7ImlkIjo4LCJ0eXBlIjoiUmFuZG9tTm9pc2UiLCJwb3MiOlswLDEwNDBdLCJzaXplIjpb"
    "NDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6NCwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6Ik5PSVNF"
    "IiwidHlwZSI6Ik5PSVNFIiwibGlua3MiOls3XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiUmFuZG9tTm9p"
    "c2UifSwid2lkZ2V0c192YWx1ZXMiOls0Ml19LHsiaWQiOjksInR5cGUiOiJCYXNpY0d1aWRlciIsInBvcyI6WzEzODAsMF0sInNp"
    "emUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVyIjoxMCwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibW9kZWwiLCJ0eXBl"
    "IjoiTU9ERUwiLCJsaW5rIjo1fSx7Im5hbWUiOiJjb25kaXRpb25pbmciLCJ0eXBlIjoiQ09ORElUSU9OSU5HIiwibGluayI6Nn1d"
    "LCJvdXRwdXRzIjpbeyJuYW1lIjoiR1VJREVSIiwidHlwZSI6IkdVSURFUiIsImxpbmtzIjpbOF19XSwicHJvcGVydGllcyI6eyJO"
    "b2RlIG5hbWUgZm9yIFMmUiI6IkJhc2ljR3VpZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbXX0seyJpZCI6MTAsInR5cGUiOiJLU2Ft"
    "cGxlclNlbGVjdCIsInBvcyI6WzAsMTMwMF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjo1LCJtb2RlIjowLCJp"
    "bnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiU0FNUExFUiIsInR5cGUiOiJTQU1QTEVSIiwibGlua3MiOls5XX1dLCJwcm9w"
    "ZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiS1NhbXBsZXJTZWxlY3QifSwid2lkZ2V0c192YWx1ZXMiOlsiZXVsZXIiXX0s"
    "eyJpZCI6MTEsInR5cGUiOiJNYW51YWxTaWdtYXMiLCJwb3MiOlswLDE1NjBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6NiwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IlNJR01BUyIsInR5cGUiOiJTSUdNQVMiLCJs"
    "aW5rcyI6WzEwXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiTWFudWFsU2lnbWFzIn0sIndpZGdldHNfdmFs"
    "dWVzIjpbIjAuOTk5OTE2NiwgMC45NzI4MzI2LCAwLjkyMzA3NjksIDAuOCwgMC4wIl19LHsiaWQiOjEyLCJ0eXBlIjoiU2FtcGxl"
    "ckN1c3RvbUFkdmFuY2VkIiwicG9zIjpbMTg0MCwwXSwic2l6ZSI6WzQwMCwxNjBdLCJmbGFncyI6e30sIm9yZGVyIjoxMSwibW9k"
    "ZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoibm9pc2UiLCJ0eXBlIjoiTk9JU0UiLCJsaW5rIjo3fSx7Im5hbWUiOiJndWlkZXIiLCJ0"
    "eXBlIjoiR1VJREVSIiwibGluayI6OH0seyJuYW1lIjoic2FtcGxlciIsInR5cGUiOiJTQU1QTEVSIiwibGluayI6OX0seyJuYW1l"
    "Ijoic2lnbWFzIiwidHlwZSI6IlNJR01BUyIsImxpbmsiOjEwfSx7Im5hbWUiOiJsYXRlbnRfaW1hZ2UiLCJ0eXBlIjoiTEFURU5U"
    "IiwibGluayI6MTF9XSwib3V0cHV0cyI6W3sibmFtZSI6Im91dHB1dCIsInR5cGUiOiJMQVRFTlQiLCJsaW5rcyI6WzEyLDE0XX0s"
    "eyJuYW1lIjoiZGVub2lzZWRfb3V0cHV0IiwidHlwZSI6IkxBVEVOVCIsImxpbmtzIjpudWxsfV0sInByb3BlcnRpZXMiOnsiTm9k"
    "ZSBuYW1lIGZvciBTJlIiOiJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7ImlkIjoxMywidHlw"
    "ZSI6IlZBRURlY29kZSIsInBvcyI6WzIzMDAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVyIjoxMiwibW9kZSI6"
    "MCwiaW5wdXRzIjpbeyJuYW1lIjoic2FtcGxlcyIsInR5cGUiOiJMQVRFTlQiLCJsaW5rIjoxMn0seyJuYW1lIjoidmFlIiwidHlw"
    "ZSI6IlZBRSIsImxpbmsiOjEzfV0sIm91dHB1dHMiOlt7Im5hbWUiOiJJTUFHRSIsInR5cGUiOiJJTUFHRSIsImxpbmtzIjpbMTZd"
    "fV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJWQUVEZWNvZGUifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7Imlk"
    "IjoxNCwidHlwZSI6IlZBRURlY29kZUF1ZGlvIiwicG9zIjpbMjMwMCwyNjBdLCJzaXplIjpbNDAwLDgyXSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6MTMsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6InNhbXBsZXMiLCJ0eXBlIjoiTEFURU5UIiwibGluayI6MTR9LHsi"
    "bmFtZSI6InZhZSIsInR5cGUiOiJWQUUiLCJsaW5rIjoxNX1dLCJvdXRwdXRzIjpbeyJuYW1lIjoiQVVESU8iLCJ0eXBlIjoiQVVE"
    "SU8iLCJsaW5rcyI6WzE3XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFFRGVjb2RlQXVkaW8ifSwid2lk"
    "Z2V0c192YWx1ZXMiOltdfSx7ImlkIjoxNSwidHlwZSI6IkNyZWF0ZVZpZGVvIiwicG9zIjpbMjc2MCwwXSwic2l6ZSI6WzQwMCwx"
    "NjBdLCJmbGFncyI6e30sIm9yZGVyIjoxNCwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoiaW1hZ2VzIiwidHlwZSI6IklNQUdF"
    "IiwibGluayI6MTZ9LHsibmFtZSI6ImF1ZGlvIiwidHlwZSI6IkFVRElPIiwibGluayI6MTcsInNoYXBlIjo3fV0sIm91dHB1dHMi"
    "Olt7Im5hbWUiOiJWSURFTyIsInR5cGUiOiJWSURFTyIsImxpbmtzIjpbMThdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZv"
    "ciBTJlIiOiJDcmVhdGVWaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WzI0LjAsOCwic1JHQiJdfSx7ImlkIjoxNiwidHlwZSI6IlNh"
    "dmVWaWRlbyIsInBvcyI6WzMyMjAsMF0sInNpemUiOls0MDAsMTM0XSwiZmxhZ3MiOnt9LCJvcmRlciI6MTUsIm1vZGUiOjAsImlu"
    "cHV0cyI6W3sibmFtZSI6InZpZGVvIiwidHlwZSI6IlZJREVPIiwibGluayI6MTh9XSwib3V0cHV0cyI6W3sibmFtZSI6InZpZGVv"
    "IiwidHlwZSI6IlZJREVPIiwibGlua3MiOm51bGx9XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlNhdmVWaWRl"
    "byJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJGYXN0SDNfU2F5YWthX1QyVkEvMDNfRmFzdEgzX1ZTQV80c3RlcCIsImF1dG8iLCJhdXRv"
    "Il19XSwibGlua3MiOltbMSwxLDAsMiwwLCJNT0RFTCJdLFsyLDIsMCwzLDAsIk1PREVMIl0sWzMsNCwwLDcsMCwiQ0xJUCJdLFs0"
    "LDUsMCw3LDEsIlZBRSJdLFs1LDMsMCw5LDAsIk1PREVMIl0sWzYsNywwLDksMSwiQ09ORElUSU9OSU5HIl0sWzcsOCwwLDEyLDAs"
    "Ik5PSVNFIl0sWzgsOSwwLDEyLDEsIkdVSURFUiJdLFs5LDEwLDAsMTIsMiwiU0FNUExFUiJdLFsxMCwxMSwwLDEyLDMsIlNJR01B"
    "UyJdLFsxMSw3LDEsMTIsNCwiTEFURU5UIl0sWzEyLDEyLDAsMTMsMCwiTEFURU5UIl0sWzEzLDUsMCwxMywxLCJWQUUiXSxbMTQs"
    "MTIsMCwxNCwwLCJMQVRFTlQiXSxbMTUsNiwwLDE0LDEsIlZBRSJdLFsxNiwxMywwLDE1LDAsIklNQUdFIl0sWzE3LDE0LDAsMTUs"
    "MSwiQVVESU8iXSxbMTgsMTUsMCwxNiwwLCJWSURFTyJdXSwiZ3JvdXBzIjpbXSwiY29uZmlnIjp7fSwiZXh0cmEiOnsiZHMiOnsi"
    "c2NhbGUiOjAuNywib2Zmc2V0IjpbMCwwXX19LCJ2ZXJzaW9uIjowLjR9"
)
_WF_I2V_B64 = (
    "eyJpZCI6ImZhc3RoMy1pMnYiLCJyZXZpc2lvbiI6MCwibGFzdF9ub2RlX2lkIjoxNywibGFzdF9saW5rX2lkIjoxOSwibm9kZXMi"
    "Olt7ImlkIjoxLCJ0eXBlIjoiVU5FVExvYWRlciIsInBvcyI6WzAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVy"
    "IjowLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6"
    "WzFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJVTkVUTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbIm1p"
    "bmltYXhfaDNfZmFzdHZpZGVvX3ZzYV9kYXRhZnJlZV8xMzAwc3RlcF80c3RlcF9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLCJk"
    "ZWZhdWx0Il19LHsiaWQiOjIsInR5cGUiOiJNaW5pTWF4SDNTaWdtYVNoaWZ0IiwicG9zIjpbNDYwLDBdLCJzaXplIjpbNDAwLDEw"
    "OF0sImZsYWdzIjp7fSwib3JkZXIiOjgsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6Im1vZGVsIiwidHlwZSI6Ik1PREVMIiwi"
    "bGluayI6MX1dLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6WzJdfV0sInByb3BlcnRp"
    "ZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJNaW5pTWF4SDNTaWdtYVNoaWZ0In0sIndpZGdldHNfdmFsdWVzIjpbMTIuMCwzLjBd"
    "fSx7ImlkIjozLCJ0eXBlIjoiU29sQXR0bk1pbmlNYXgiLCJwb3MiOls5MjAsMF0sInNpemUiOls0MDAsMjEyXSwiZmxhZ3MiOnt9"
    "LCJvcmRlciI6MTAsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6Im1vZGVsIiwidHlwZSI6Ik1PREVMIiwibGluayI6Mn1dLCJv"
    "dXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6WzZdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBu"
    "YW1lIGZvciBTJlIiOiJTb2xBdHRuTWluaU1heCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJWU0EgKEZhc3RWaWRlbykiLDAuMCwxLjAs"
    "MCwiZXhhY3Rfa3ZfYW5kX3Jvd3MiLHRydWVdfSx7ImlkIjo0LCJ0eXBlIjoiQ0xJUExvYWRlciIsInBvcyI6WzAsMjYwXSwic2l6"
    "ZSI6WzQwMCwxMDhdLCJmbGFncyI6e30sIm9yZGVyIjoxLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoi"
    "Q0xJUCIsInR5cGUiOiJDTElQIiwibGlua3MiOlszXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiQ0xJUExv"
    "YWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJxd2VuM3ZsXzMyYl9taW5pbWF4X2gzX252ZnA0X2F3cS5zYWZldGVuc29ycyIsIm1p"
    "bmltYXgiLCJkZWZhdWx0Il19LHsiaWQiOjUsInR5cGUiOiJWQUVMb2FkZXIiLCJwb3MiOlswLDUyMF0sInNpemUiOls0MDAsNjBd"
    "LCJmbGFncyI6e30sIm9yZGVyIjoyLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6"
    "IlZBRSIsImxpbmtzIjpbNCwxNF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRn"
    "ZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX3ZpZGVvX3ZhZV9mcDE2LnNhZmV0ZW5zb3JzIl19LHsiaWQiOjYsInR5cGUiOiJWQUVM"
    "b2FkZXIiLCJwb3MiOlswLDc4MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjozLCJtb2RlIjowLCJpbnB1dHMi"
    "OltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6IlZBRSIsImxpbmtzIjpbMTZdfV0sInByb3BlcnRpZXMiOnsiTm9k"
    "ZSBuYW1lIGZvciBTJlIiOiJWQUVMb2FkZXIifSwid2lkZ2V0c192YWx1ZXMiOlsibWluaW1heF9oM19hdWRpb192YWVfZnAzMi5z"
    "YWZldGVuc29ycyJdfSx7ImlkIjo3LCJ0eXBlIjoiTWluaU1heEgzSW1hZ2VUb1ZpZGVvIiwicG9zIjpbNDYwLDI2MF0sInNpemUi"
    "Ols0MDAsMjM4XSwiZmxhZ3MiOnt9LCJvcmRlciI6OSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoiY2xpcCIsInR5cGUiOiJD"
    "TElQIiwibGluayI6M30seyJuYW1lIjoidmFlIiwidHlwZSI6IlZBRSIsImxpbmsiOjR9LHsibmFtZSI6ImZpcnN0X2ZyYW1lIiwi"
    "dHlwZSI6IklNQUdFIiwibGluayI6NSwic2hhcGUiOjd9LHsibmFtZSI6Imxhc3RfZnJhbWUiLCJ0eXBlIjoiSU1BR0UiLCJsaW5r"
    "IjpudWxsLCJzaGFwZSI6N31dLCJvdXRwdXRzIjpbeyJuYW1lIjoicG9zaXRpdmUiLCJ0eXBlIjoiQ09ORElUSU9OSU5HIiwibGlu"
    "a3MiOls3XX0seyJuYW1lIjoiTEFURU5UIiwidHlwZSI6IkxBVEVOVCIsImxpbmtzIjpbMTJdfV0sInByb3BlcnRpZXMiOnsiTm9k"
    "ZSBuYW1lIGZvciBTJlIiOiJNaW5pTWF4SDNJbWFnZVRvVmlkZW8ifSwid2lkZ2V0c192YWx1ZXMiOlsiaW50ZWdyYXRlZF9tdWx0"
    "aW1vZGFsX2Rlc2NyaXB0aW9uOiBbU2hvdCAxXSAyRC1hbmltYXRlZCwgYW5pbWUgc3R5bGUsIGEgbWVkaXVtIGNsb3NlLXVwIHNo"
    "b3QgZnJhbWVzIGEgY2hlZXJmdWwgeW91bmcgd29tYW4gd2l0aCBkYXJrIGhhaXIgdGllZCB1cCBpbiBhIG5lYXQgYnVuLCBzaWRl"
    "IGJhbmdzIGZyYW1pbmcgaGVyIGRlbGljYXRlIGZhY2UsIGFuZCBkYXJrIHB1cnBsZSBleWVzIHdlYXJpbmcgc21hbGwgcGVhcmwg"
    "ZWFycmluZ3MuIFNoZSB3ZWFycyBhIGxpZ2h0IGdyYXkgdHVydGxlbmVjayBrbml0IHRvcCB1bmRlciBhIGJsYWNrIHNwYWdoZXR0"
    "aS1zdHJhcCBtaW5pIGRyZXNzLiBTdGFuZGluZyBpbiBhIGNsZWFuIG1vZGVybiByb29tIHdpdGggd2hpdGUgd2FsbHMgYW5kIGEg"
    "ZGFyayBkb29yZnJhbWUsIHNoZSB0aWx0cyBoZXIgaGVhZCBzbGlnaHRseSB3aXRoIGEgYnJpZ2h0IHNtaWxlLCBsaWdodGx5IHJl"
    "c3RpbmcgaGVyIHJpZ2h0IGZpbmdlcnRpcHMgbmVhciBoZXIgY29sbGFyYm9uZS4gVGhlIGNhbWVyYSBwdXNoZXMgaW4gd2l0aCBz"
    "bWFsbCBhbXBsaXR1ZGUgYXQgc2xvdyBzcGVlZCBhcyB0aGUgeW91bmcgd29tYW4gd2l0aCBhIHN3ZWV0LCBjbGVhciB2b2ljZSAo"
    "UzEpIGdlbnRseSBzYXlzOiA8ZD5bSmFwYW5lc2VdIOOBk+OCk+OBq+OBoeOBr++8geS7iuaXpeOCguS4gOaXpemgkeW8teOCjeOB"
    "huOBreOAgjwvZD5cblxub3ZlcmFsbF9zb3VuZHNjYXBlOiBRdWlldCBpbmRvb3Igcm9vbSB0b25lLCBhIHNvZnQgcnVzdGxlIG9m"
    "IGtuaXQgZmFicmljIGFzIHNoZSBtb3ZlcyBoZXIgc2hvdWxkZXIsIGFuZCB0aGUgY2xlYXIgc3Bva2VuIHZvaWNlIGluIGEgbmF0"
    "dXJhbCByb29tIGFjb3VzdGljLlxuXG5ub25fZGllZ2V0aWNfbXVzaWM6IEEgZ2VudGxlLCB1cGxpZnRpbmcgYWNvdXN0aWMgZ3Vp"
    "dGFyIHBhdHRlcm4gd2l0aCB3YXJtIHBpYW5vIG5vdGVzIGF0IGEgbW9kZXJhdGUgdGVtcG8sIGZhZGluZyBzb2Z0bHkgYXQgdGhl"
    "IGVuZC4iLDU0NCw4MzIsMTI0XX0seyJpZCI6OCwidHlwZSI6IlJhbmRvbU5vaXNlIiwicG9zIjpbMCwxMDQwXSwic2l6ZSI6WzQw"
    "MCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjQsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJOT0lTRSIs"
    "InR5cGUiOiJOT0lTRSIsImxpbmtzIjpbOF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlJhbmRvbU5vaXNl"
    "In0sIndpZGdldHNfdmFsdWVzIjpbNDJdfSx7ImlkIjo5LCJ0eXBlIjoiQmFzaWNHdWlkZXIiLCJwb3MiOlsxMzgwLDBdLCJzaXpl"
    "IjpbNDAwLDgyXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTEsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6Im1vZGVsIiwidHlwZSI6"
    "Ik1PREVMIiwibGluayI6Nn0seyJuYW1lIjoiY29uZGl0aW9uaW5nIiwidHlwZSI6IkNPTkRJVElPTklORyIsImxpbmsiOjd9XSwi"
    "b3V0cHV0cyI6W3sibmFtZSI6IkdVSURFUiIsInR5cGUiOiJHVUlERVIiLCJsaW5rcyI6WzldfV0sInByb3BlcnRpZXMiOnsiTm9k"
    "ZSBuYW1lIGZvciBTJlIiOiJCYXNpY0d1aWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6W119LHsiaWQiOjEwLCJ0eXBlIjoiS1NhbXBs"
    "ZXJTZWxlY3QiLCJwb3MiOlswLDEzMDBdLCJzaXplIjpbNDAwLDYwXSwiZmxhZ3MiOnt9LCJvcmRlciI6NSwibW9kZSI6MCwiaW5w"
    "dXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IlNBTVBMRVIiLCJ0eXBlIjoiU0FNUExFUiIsImxpbmtzIjpbMTBdfV0sInByb3Bl"
    "cnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJLU2FtcGxlclNlbGVjdCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJldWxlciJdfSx7"
    "ImlkIjoxMSwidHlwZSI6Ik1hbnVhbFNpZ21hcyIsInBvcyI6WzAsMTU2MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9y"
    "ZGVyIjo2LCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiU0lHTUFTIiwidHlwZSI6IlNJR01BUyIsImxp"
    "bmtzIjpbMTFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJNYW51YWxTaWdtYXMifSwid2lkZ2V0c192YWx1"
    "ZXMiOlsiMC45OTk5MTY2LCAwLjk3MjgzMjYsIDAuOTIzMDc2OSwgMC44LCAwLjAiXX0seyJpZCI6MTIsInR5cGUiOiJTYW1wbGVy"
    "Q3VzdG9tQWR2YW5jZWQiLCJwb3MiOlsxODQwLDBdLCJzaXplIjpbNDAwLDE2MF0sImZsYWdzIjp7fSwib3JkZXIiOjEyLCJtb2Rl"
    "IjowLCJpbnB1dHMiOlt7Im5hbWUiOiJub2lzZSIsInR5cGUiOiJOT0lTRSIsImxpbmsiOjh9LHsibmFtZSI6Imd1aWRlciIsInR5"
    "cGUiOiJHVUlERVIiLCJsaW5rIjo5fSx7Im5hbWUiOiJzYW1wbGVyIiwidHlwZSI6IlNBTVBMRVIiLCJsaW5rIjoxMH0seyJuYW1l"
    "Ijoic2lnbWFzIiwidHlwZSI6IlNJR01BUyIsImxpbmsiOjExfSx7Im5hbWUiOiJsYXRlbnRfaW1hZ2UiLCJ0eXBlIjoiTEFURU5U"
    "IiwibGluayI6MTJ9XSwib3V0cHV0cyI6W3sibmFtZSI6Im91dHB1dCIsInR5cGUiOiJMQVRFTlQiLCJsaW5rcyI6WzEzLDE1XX0s"
    "eyJuYW1lIjoiZGVub2lzZWRfb3V0cHV0IiwidHlwZSI6IkxBVEVOVCIsImxpbmtzIjpudWxsfV0sInByb3BlcnRpZXMiOnsiTm9k"
    "ZSBuYW1lIGZvciBTJlIiOiJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7ImlkIjoxMywidHlw"
    "ZSI6IlZBRURlY29kZSIsInBvcyI6WzIzMDAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVyIjoxMywibW9kZSI6"
    "MCwiaW5wdXRzIjpbeyJuYW1lIjoic2FtcGxlcyIsInR5cGUiOiJMQVRFTlQiLCJsaW5rIjoxM30seyJuYW1lIjoidmFlIiwidHlw"
    "ZSI6IlZBRSIsImxpbmsiOjE0fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJJTUFHRSIsInR5cGUiOiJJTUFHRSIsImxpbmtzIjpbMTdd"
    "fV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJWQUVEZWNvZGUifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7Imlk"
    "IjoxNCwidHlwZSI6IlZBRURlY29kZUF1ZGlvIiwicG9zIjpbMjMwMCwyNjBdLCJzaXplIjpbNDAwLDgyXSwiZmxhZ3MiOnt9LCJv"
    "cmRlciI6MTQsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6InNhbXBsZXMiLCJ0eXBlIjoiTEFURU5UIiwibGluayI6MTV9LHsi"
    "bmFtZSI6InZhZSIsInR5cGUiOiJWQUUiLCJsaW5rIjoxNn1dLCJvdXRwdXRzIjpbeyJuYW1lIjoiQVVESU8iLCJ0eXBlIjoiQVVE"
    "SU8iLCJsaW5rcyI6WzE4XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiVkFFRGVjb2RlQXVkaW8ifSwid2lk"
    "Z2V0c192YWx1ZXMiOltdfSx7ImlkIjoxNSwidHlwZSI6IkNyZWF0ZVZpZGVvIiwicG9zIjpbMjc2MCwwXSwic2l6ZSI6WzQwMCwx"
    "NjBdLCJmbGFncyI6e30sIm9yZGVyIjoxNSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoiaW1hZ2VzIiwidHlwZSI6IklNQUdF"
    "IiwibGluayI6MTd9LHsibmFtZSI6ImF1ZGlvIiwidHlwZSI6IkFVRElPIiwibGluayI6MTgsInNoYXBlIjo3fV0sIm91dHB1dHMi"
    "Olt7Im5hbWUiOiJWSURFTyIsInR5cGUiOiJWSURFTyIsImxpbmtzIjpbMTldfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZv"
    "ciBTJlIiOiJDcmVhdGVWaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WzI0LjAsOCwic1JHQiJdfSx7ImlkIjoxNiwidHlwZSI6IlNh"
    "dmVWaWRlbyIsInBvcyI6WzMyMjAsMF0sInNpemUiOls0MDAsMTM0XSwiZmxhZ3MiOnt9LCJvcmRlciI6MTYsIm1vZGUiOjAsImlu"
    "cHV0cyI6W3sibmFtZSI6InZpZGVvIiwidHlwZSI6IlZJREVPIiwibGluayI6MTl9XSwib3V0cHV0cyI6W3sibmFtZSI6InZpZGVv"
    "IiwidHlwZSI6IlZJREVPIiwibGlua3MiOm51bGx9XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlNhdmVWaWRl"
    "byJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJGYXN0SDMvZmFzdGgzX2kydiIsImF1dG8iLCJhdXRvIl19LHsiaWQiOjE3LCJ0eXBlIjoi"
    "TG9hZEltYWdlIiwicG9zIjpbMCwxODIwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjcsIm1vZGUiOjAsImlu"
    "cHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJJTUFHRSIsInR5cGUiOiJJTUFHRSIsImxpbmtzIjpbNV19LHsibmFtZSI6Ik1B"
    "U0siLCJ0eXBlIjoiTUFTSyIsImxpbmtzIjpudWxsfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJMb2FkSW1h"
    "Z2UifSwid2lkZ2V0c192YWx1ZXMiOlsic2cyNl9JMlZfaW1hZ2VfU0FNUExFX21vb24ucG5nIl19XSwibGlua3MiOltbMSwxLDAs"
    "MiwwLCJNT0RFTCJdLFsyLDIsMCwzLDAsIk1PREVMIl0sWzMsNCwwLDcsMCwiQ0xJUCJdLFs0LDUsMCw3LDEsIlZBRSJdLFs1LDE3"
    "LDAsNywyLCJJTUFHRSJdLFs2LDMsMCw5LDAsIk1PREVMIl0sWzcsNywwLDksMSwiQ09ORElUSU9OSU5HIl0sWzgsOCwwLDEyLDAs"
    "Ik5PSVNFIl0sWzksOSwwLDEyLDEsIkdVSURFUiJdLFsxMCwxMCwwLDEyLDIsIlNBTVBMRVIiXSxbMTEsMTEsMCwxMiwzLCJTSUdN"
    "QVMiXSxbMTIsNywxLDEyLDQsIkxBVEVOVCJdLFsxMywxMiwwLDEzLDAsIkxBVEVOVCJdLFsxNCw1LDAsMTMsMSwiVkFFIl0sWzE1"
    "LDEyLDAsMTQsMCwiTEFURU5UIl0sWzE2LDYsMCwxNCwxLCJWQUUiXSxbMTcsMTMsMCwxNSwwLCJJTUFHRSJdLFsxOCwxNCwwLDE1"
    "LDEsIkFVRElPIl0sWzE5LDE1LDAsMTYsMCwiVklERU8iXV0sImdyb3VwcyI6W10sImNvbmZpZyI6e30sImV4dHJhIjp7ImRzIjp7"
    "InNjYWxlIjowLjcsIm9mZnNldCI6WzAsMF19fSwidmVyc2lvbiI6MC40fQ=="
)
_WF_R2V_B64 = (
    "eyJpZCI6ImZhc3RoMy1yMnYiLCJyZXZpc2lvbiI6MCwibGFzdF9ub2RlX2lkIjoxNywibGFzdF9saW5rX2lkIjoxOSwibm9kZXMi"
    "Olt7ImlkIjoxLCJ0eXBlIjoiVU5FVExvYWRlciIsInBvcyI6WzAsMF0sInNpemUiOls0MDAsODJdLCJmbGFncyI6e30sIm9yZGVy"
    "IjowLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6"
    "WzFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJVTkVUTG9hZGVyIn0sIndpZGdldHNfdmFsdWVzIjpbIm1p"
    "bmltYXhfaDNfZmFzdHZpZGVvX3ZzYV9kYXRhZnJlZV8xMzAwc3RlcF80c3RlcF9pbnQ4X2NvbnZyb3Quc2FmZXRlbnNvcnMiLCJk"
    "ZWZhdWx0Il19LHsiaWQiOjIsInR5cGUiOiJNaW5pTWF4SDNTaWdtYVNoaWZ0IiwicG9zIjpbNDYwLDBdLCJzaXplIjpbNDAwLDEw"
    "OF0sImZsYWdzIjp7fSwib3JkZXIiOjgsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6Im1vZGVsIiwidHlwZSI6Ik1PREVMIiwi"
    "bGluayI6MX1dLCJvdXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6WzJdfV0sInByb3BlcnRp"
    "ZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJNaW5pTWF4SDNTaWdtYVNoaWZ0In0sIndpZGdldHNfdmFsdWVzIjpbMTIuMCwzLjBd"
    "fSx7ImlkIjozLCJ0eXBlIjoiU29sQXR0bk1pbmlNYXgiLCJwb3MiOls5MjAsMF0sInNpemUiOls0MDAsMjEyXSwiZmxhZ3MiOnt9"
    "LCJvcmRlciI6MTAsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6Im1vZGVsIiwidHlwZSI6Ik1PREVMIiwibGluayI6Mn1dLCJv"
    "dXRwdXRzIjpbeyJuYW1lIjoiTU9ERUwiLCJ0eXBlIjoiTU9ERUwiLCJsaW5rcyI6WzZdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBu"
    "YW1lIGZvciBTJlIiOiJTb2xBdHRuTWluaU1heCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJWU0EgKEZhc3RWaWRlbykiLDAuMCwxLjAs"
    "MCwiZXhhY3Rfa3ZfYW5kX3Jvd3MiLHRydWVdfSx7ImlkIjo0LCJ0eXBlIjoiQ0xJUExvYWRlciIsInBvcyI6WzAsMjYwXSwic2l6"
    "ZSI6WzQwMCwxMDhdLCJmbGFncyI6e30sIm9yZGVyIjoxLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoi"
    "Q0xJUCIsInR5cGUiOiJDTElQIiwibGlua3MiOlszXX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiQ0xJUExv"
    "YWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJxd2VuM3ZsXzMyYl9taW5pbWF4X2gzX252ZnA0X2F3cS5zYWZldGVuc29ycyIsIm1p"
    "bmltYXgiLCJkZWZhdWx0Il19LHsiaWQiOjUsInR5cGUiOiJWQUVMb2FkZXIiLCJwb3MiOlswLDUyMF0sInNpemUiOls0MDAsNjBd"
    "LCJmbGFncyI6e30sIm9yZGVyIjoyLCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6"
    "IlZBRSIsImxpbmtzIjpbNCwxNF19XSwicHJvcGVydGllcyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRn"
    "ZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX3ZpZGVvX3ZhZV9mcDE2LnNhZmV0ZW5zb3JzIl19LHsiaWQiOjYsInR5cGUiOiJWQUVM"
    "b2FkZXIiLCJwb3MiOlswLDc4MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjozLCJtb2RlIjowLCJpbnB1dHMi"
    "OltdLCJvdXRwdXRzIjpbeyJuYW1lIjoiVkFFIiwidHlwZSI6IlZBRSIsImxpbmtzIjpbNSwxNl19XSwicHJvcGVydGllcyI6eyJO"
    "b2RlIG5hbWUgZm9yIFMmUiI6IlZBRUxvYWRlciJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJtaW5pbWF4X2gzX2F1ZGlvX3ZhZV9mcDMy"
    "LnNhZmV0ZW5zb3JzIl19LHsiaWQiOjcsInR5cGUiOiJNaW5pTWF4SDNSZWZlcmVuY2VUb1ZpZGVvIiwicG9zIjpbNDYwLDI2MF0s"
    "InNpemUiOls0MDAsMzE2XSwiZmxhZ3MiOnt9LCJvcmRlciI6OSwibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoiY2xpcCIsInR5"
    "cGUiOiJDTElQIiwibGluayI6M30seyJuYW1lIjoidmFlIiwidHlwZSI6IlZBRSIsImxpbmsiOjR9LHsibmFtZSI6ImF1ZGlvX3Zh"
    "ZSIsInR5cGUiOiJWQUUiLCJsaW5rIjo1fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJwb3NpdGl2ZSIsInR5cGUiOiJDT05ESVRJT05J"
    "TkciLCJsaW5rcyI6WzddfSx7Im5hbWUiOiJMQVRFTlQiLCJ0eXBlIjoiTEFURU5UIiwibGlua3MiOlsxMl19XSwicHJvcGVydGll"
    "cyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6Ik1pbmlNYXhIM1JlZmVyZW5jZVRvVmlkZW8ifSwid2lkZ2V0c192YWx1ZXMiOlsiaW50"
    "ZWdyYXRlZF9tdWx0aW1vZGFsX2Rlc2NyaXB0aW9uOiBbU2hvdCAxXSAyRC1hbmltYXRlZCwgYW5pbWUgc3R5bGUsIGEgbWVkaXVt"
    "IGNsb3NlLXVwIHNob3QgZnJhbWVzIGEgY2hlZXJmdWwgeW91bmcgd29tYW4gd2l0aCBkYXJrIGhhaXIgdGllZCB1cCBpbiBhIG5l"
    "YXQgYnVuLCBzaWRlIGJhbmdzIGZyYW1pbmcgaGVyIGRlbGljYXRlIGZhY2UsIGFuZCBkYXJrIHB1cnBsZSBleWVzIHdlYXJpbmcg"
    "c21hbGwgcGVhcmwgZWFycmluZ3MuIFNoZSB3ZWFycyBhIGxpZ2h0IGdyYXkgdHVydGxlbmVjayBrbml0IHRvcCB1bmRlciBhIGJs"
    "YWNrIHNwYWdoZXR0aS1zdHJhcCBtaW5pIGRyZXNzLiBTdGFuZGluZyBpbiBhIGNsZWFuIG1vZGVybiByb29tIHdpdGggd2hpdGUg"
    "d2FsbHMgYW5kIGEgZGFyayBkb29yZnJhbWUsIHNoZSB0aWx0cyBoZXIgaGVhZCBzbGlnaHRseSB3aXRoIGEgYnJpZ2h0IHNtaWxl"
    "LCBsaWdodGx5IHJlc3RpbmcgaGVyIHJpZ2h0IGZpbmdlcnRpcHMgbmVhciBoZXIgY29sbGFyYm9uZS4gVGhlIGNhbWVyYSBwdXNo"
    "ZXMgaW4gd2l0aCBzbWFsbCBhbXBsaXR1ZGUgYXQgc2xvdyBzcGVlZCBhcyB0aGUgeW91bmcgd29tYW4gd2l0aCBhIHN3ZWV0LCBj"
    "bGVhciB2b2ljZSAoUzEpIGdlbnRseSBzYXlzOiA8ZD5bSmFwYW5lc2VdIOOBk+OCk+OBq+OBoeOBr++8geS7iuaXpeOCguS4gOaX"
    "pemgkeW8teOCjeOBhuOBreOAgjwvZD5cblxub3ZlcmFsbF9zb3VuZHNjYXBlOiBRdWlldCBpbmRvb3Igcm9vbSB0b25lLCBhIHNv"
    "ZnQgcnVzdGxlIG9mIGtuaXQgZmFicmljIGFzIHNoZSBtb3ZlcyBoZXIgc2hvdWxkZXIsIGFuZCB0aGUgY2xlYXIgc3Bva2VuIHZv"
    "aWNlIGluIGEgbmF0dXJhbCByb29tIGFjb3VzdGljLlxuXG5ub25fZGllZ2V0aWNfbXVzaWM6IEEgZ2VudGxlLCB1cGxpZnRpbmcg"
    "YWNvdXN0aWMgZ3VpdGFyIHBhdHRlcm4gd2l0aCB3YXJtIHBpYW5vIG5vdGVzIGF0IGEgbW9kZXJhdGUgdGVtcG8sIGZhZGluZyBz"
    "b2Z0bHkgYXQgdGhlIGVuZC4iLDU0NCw4MzIsMTI0LCJtYXgiLG51bGwsbnVsbCxudWxsXX0seyJpZCI6OCwidHlwZSI6IlJhbmRv"
    "bU5vaXNlIiwicG9zIjpbMCwxMDQwXSwic2l6ZSI6WzQwMCw2MF0sImZsYWdzIjp7fSwib3JkZXIiOjQsIm1vZGUiOjAsImlucHV0"
    "cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJOT0lTRSIsInR5cGUiOiJOT0lTRSIsImxpbmtzIjpbOF19XSwicHJvcGVydGllcyI6"
    "eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlJhbmRvbU5vaXNlIn0sIndpZGdldHNfdmFsdWVzIjpbNDJdfSx7ImlkIjo5LCJ0eXBlIjoi"
    "QmFzaWNHdWlkZXIiLCJwb3MiOlsxMzgwLDBdLCJzaXplIjpbNDAwLDgyXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTEsIm1vZGUiOjAs"
    "ImlucHV0cyI6W3sibmFtZSI6Im1vZGVsIiwidHlwZSI6Ik1PREVMIiwibGluayI6Nn0seyJuYW1lIjoiY29uZGl0aW9uaW5nIiwi"
    "dHlwZSI6IkNPTkRJVElPTklORyIsImxpbmsiOjd9XSwib3V0cHV0cyI6W3sibmFtZSI6IkdVSURFUiIsInR5cGUiOiJHVUlERVIi"
    "LCJsaW5rcyI6WzldfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJCYXNpY0d1aWRlciJ9LCJ3aWRnZXRzX3Zh"
    "bHVlcyI6W119LHsiaWQiOjEwLCJ0eXBlIjoiS1NhbXBsZXJTZWxlY3QiLCJwb3MiOlswLDEzMDBdLCJzaXplIjpbNDAwLDYwXSwi"
    "ZmxhZ3MiOnt9LCJvcmRlciI6NSwibW9kZSI6MCwiaW5wdXRzIjpbXSwib3V0cHV0cyI6W3sibmFtZSI6IlNBTVBMRVIiLCJ0eXBl"
    "IjoiU0FNUExFUiIsImxpbmtzIjpbMTBdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJLU2FtcGxlclNlbGVj"
    "dCJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJldWxlciJdfSx7ImlkIjoxMSwidHlwZSI6Ik1hbnVhbFNpZ21hcyIsInBvcyI6WzAsMTU2"
    "MF0sInNpemUiOls0MDAsNjBdLCJmbGFncyI6e30sIm9yZGVyIjo2LCJtb2RlIjowLCJpbnB1dHMiOltdLCJvdXRwdXRzIjpbeyJu"
    "YW1lIjoiU0lHTUFTIiwidHlwZSI6IlNJR01BUyIsImxpbmtzIjpbMTFdfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBT"
    "JlIiOiJNYW51YWxTaWdtYXMifSwid2lkZ2V0c192YWx1ZXMiOlsiMC45OTk5MTY2LCAwLjk3MjgzMjYsIDAuOTIzMDc2OSwgMC44"
    "LCAwLjAiXX0seyJpZCI6MTIsInR5cGUiOiJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQiLCJwb3MiOlsxODQwLDBdLCJzaXplIjpbNDAw"
    "LDE2MF0sImZsYWdzIjp7fSwib3JkZXIiOjEyLCJtb2RlIjowLCJpbnB1dHMiOlt7Im5hbWUiOiJub2lzZSIsInR5cGUiOiJOT0lT"
    "RSIsImxpbmsiOjh9LHsibmFtZSI6Imd1aWRlciIsInR5cGUiOiJHVUlERVIiLCJsaW5rIjo5fSx7Im5hbWUiOiJzYW1wbGVyIiwi"
    "dHlwZSI6IlNBTVBMRVIiLCJsaW5rIjoxMH0seyJuYW1lIjoic2lnbWFzIiwidHlwZSI6IlNJR01BUyIsImxpbmsiOjExfSx7Im5h"
    "bWUiOiJsYXRlbnRfaW1hZ2UiLCJ0eXBlIjoiTEFURU5UIiwibGluayI6MTJ9XSwib3V0cHV0cyI6W3sibmFtZSI6Im91dHB1dCIs"
    "InR5cGUiOiJMQVRFTlQiLCJsaW5rcyI6WzEzLDE1XX0seyJuYW1lIjoiZGVub2lzZWRfb3V0cHV0IiwidHlwZSI6IkxBVEVOVCIs"
    "ImxpbmtzIjpudWxsfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJTYW1wbGVyQ3VzdG9tQWR2YW5jZWQifSwi"
    "d2lkZ2V0c192YWx1ZXMiOltdfSx7ImlkIjoxMywidHlwZSI6IlZBRURlY29kZSIsInBvcyI6WzIzMDAsMF0sInNpemUiOls0MDAs"
    "ODJdLCJmbGFncyI6e30sIm9yZGVyIjoxMywibW9kZSI6MCwiaW5wdXRzIjpbeyJuYW1lIjoic2FtcGxlcyIsInR5cGUiOiJMQVRF"
    "TlQiLCJsaW5rIjoxM30seyJuYW1lIjoidmFlIiwidHlwZSI6IlZBRSIsImxpbmsiOjE0fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJJ"
    "TUFHRSIsInR5cGUiOiJJTUFHRSIsImxpbmtzIjpbMTddfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJWQUVE"
    "ZWNvZGUifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7ImlkIjoxNCwidHlwZSI6IlZBRURlY29kZUF1ZGlvIiwicG9zIjpbMjMwMCwy"
    "NjBdLCJzaXplIjpbNDAwLDgyXSwiZmxhZ3MiOnt9LCJvcmRlciI6MTQsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6InNhbXBs"
    "ZXMiLCJ0eXBlIjoiTEFURU5UIiwibGluayI6MTV9LHsibmFtZSI6InZhZSIsInR5cGUiOiJWQUUiLCJsaW5rIjoxNn1dLCJvdXRw"
    "dXRzIjpbeyJuYW1lIjoiQVVESU8iLCJ0eXBlIjoiQVVESU8iLCJsaW5rcyI6WzE4XX1dLCJwcm9wZXJ0aWVzIjp7Ik5vZGUgbmFt"
    "ZSBmb3IgUyZSIjoiVkFFRGVjb2RlQXVkaW8ifSwid2lkZ2V0c192YWx1ZXMiOltdfSx7ImlkIjoxNSwidHlwZSI6IkNyZWF0ZVZp"
    "ZGVvIiwicG9zIjpbMjc2MCwwXSwic2l6ZSI6WzQwMCwxNjBdLCJmbGFncyI6e30sIm9yZGVyIjoxNSwibW9kZSI6MCwiaW5wdXRz"
    "IjpbeyJuYW1lIjoiaW1hZ2VzIiwidHlwZSI6IklNQUdFIiwibGluayI6MTd9LHsibmFtZSI6ImF1ZGlvIiwidHlwZSI6IkFVRElP"
    "IiwibGluayI6MTgsInNoYXBlIjo3fV0sIm91dHB1dHMiOlt7Im5hbWUiOiJWSURFTyIsInR5cGUiOiJWSURFTyIsImxpbmtzIjpb"
    "MTldfV0sInByb3BlcnRpZXMiOnsiTm9kZSBuYW1lIGZvciBTJlIiOiJDcmVhdGVWaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WzI0"
    "LjAsOCwic1JHQiJdfSx7ImlkIjoxNiwidHlwZSI6IlNhdmVWaWRlbyIsInBvcyI6WzMyMjAsMF0sInNpemUiOls0MDAsMTM0XSwi"
    "ZmxhZ3MiOnt9LCJvcmRlciI6MTYsIm1vZGUiOjAsImlucHV0cyI6W3sibmFtZSI6InZpZGVvIiwidHlwZSI6IlZJREVPIiwibGlu"
    "ayI6MTl9XSwib3V0cHV0cyI6W3sibmFtZSI6InZpZGVvIiwidHlwZSI6IlZJREVPIiwibGlua3MiOm51bGx9XSwicHJvcGVydGll"
    "cyI6eyJOb2RlIG5hbWUgZm9yIFMmUiI6IlNhdmVWaWRlbyJ9LCJ3aWRnZXRzX3ZhbHVlcyI6WyJGYXN0SDMvZmFzdGgzX3IydiIs"
    "ImF1dG8iLCJhdXRvIl19LHsiaWQiOjE3LCJ0eXBlIjoiTG9hZEltYWdlIiwicG9zIjpbMCwxODIwXSwic2l6ZSI6WzQwMCw2MF0s"
    "ImZsYWdzIjp7fSwib3JkZXIiOjcsIm1vZGUiOjAsImlucHV0cyI6W10sIm91dHB1dHMiOlt7Im5hbWUiOiJJTUFHRSIsInR5cGUi"
    "OiJJTUFHRSIsImxpbmtzIjpudWxsfSx7Im5hbWUiOiJNQVNLIiwidHlwZSI6Ik1BU0siLCJsaW5rcyI6bnVsbH1dLCJwcm9wZXJ0"
    "aWVzIjp7Ik5vZGUgbmFtZSBmb3IgUyZSIjoiTG9hZEltYWdlIn0sIndpZGdldHNfdmFsdWVzIjpbInNnMjZfSTJWX2ltYWdlX1NB"
    "TVBMRV9tb29uLnBuZyJdfV0sImxpbmtzIjpbWzEsMSwwLDIsMCwiTU9ERUwiXSxbMiwyLDAsMywwLCJNT0RFTCJdLFszLDQsMCw3"
    "LDAsIkNMSVAiXSxbNCw1LDAsNywxLCJWQUUiXSxbNSw2LDAsNywyLCJWQUUiXSxbNiwzLDAsOSwwLCJNT0RFTCJdLFs3LDcsMCw5"
    "LDEsIkNPTkRJVElPTklORyJdLFs4LDgsMCwxMiwwLCJOT0lTRSJdLFs5LDksMCwxMiwxLCJHVUlERVIiXSxbMTAsMTAsMCwxMiwy"
    "LCJTQU1QTEVSIl0sWzExLDExLDAsMTIsMywiU0lHTUFTIl0sWzEyLDcsMSwxMiw0LCJMQVRFTlQiXSxbMTMsMTIsMCwxMywwLCJM"
    "QVRFTlQiXSxbMTQsNSwwLDEzLDEsIlZBRSJdLFsxNSwxMiwwLDE0LDAsIkxBVEVOVCJdLFsxNiw2LDAsMTQsMSwiVkFFIl0sWzE3"
    "LDEzLDAsMTUsMCwiSU1BR0UiXSxbMTgsMTQsMCwxNSwxLCJBVURJTyJdLFsxOSwxNSwwLDE2LDAsIlZJREVPIl1dLCJncm91cHMi"
    "OltdLCJjb25maWciOnt9LCJleHRyYSI6eyJkcyI6eyJzY2FsZSI6MC43LCJvZmZzZXQiOlswLDBdfX0sInZlcnNpb24iOjAuNH0="
)

_FASTH3_WFS = {"fasth3_vsa_t2v.json": _WF_T2V_B64,
               "fasth3_vsa_i2v.json": _WF_I2V_B64,
               "fasth3_vsa_r2v.json": _WF_R2V_B64}

print("")
print("📥 FastH3のワークフローを配置します（UI形式・画面から開けます）...")
for _n, _b in _FASTH3_WFS.items():
    _p = _os.path.join(h3_workflow_dir, _n)
    with open(_p, "w", encoding="utf-8") as _f:
        _f.write(_b64.b64decode(_b).decode("utf-8"))
    _wf = _json.load(open(_p, encoding="utf-8"))
    print("  ✓ %s（ノード%d / 接続%d）" % (_n, len(_wf["nodes"]), len(_wf["links"])))

# 参考用に、配布元のAPI形式サンプルもそのまま置いておく（プログラムから投げたい人向け）
_api_dir = _os.path.join(h3_workflow_dir, "api形式_参考")
_os.makedirs(_api_dir, exist_ok=True)
!wget -q -O "{_api_dir}/fasth3_vsa_sample_api.json" "https://raw.githubusercontent.com/sepiablue-ai/minimax_h3_workflows/main/fasth3_vsa_sample.json"

print("")
print("✓ ComfyUI起動後、左メニューの Workflows > User workflows > FastH3 から選べます。")
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🌱 FastH3を使うときの注意（実測にもとづく）
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
● ステップ数は 4 のままにしてください
  4ステップ専用に訓練されたモデルです。増やしても良くなりません。
  ManualSigmas の値 0.9999166, 0.9728326, 0.9230769, 0.8, 0.0 も変えないでください。

● 解像度 × 秒数には上限があります
  VRAM 24GB(L4)の目安 : メガピクセル × 秒 ≦ 約16
    → 768p(1.06MP)なら15秒まで、1080p(2.09MP)なら8秒まで
  VRAM 80GB(A100)なら、より長く・大きくできます（上限は未検証）
  超えると落ちるか、GPU使用率100%のまま永久に終わらなくなります。

● 幅と高さは 32 の倍数にしてください（1080p は 1920×1088）

● キャラクターを似せたいなら fasth3_vsa_i2v を使ってください
  4ステップでは参照画像からの再現(R2V)がほとんど効きません。
  1フレーム目に置いた画像は出力の1フレーム目そのものになるので、
  I2Vなら顔が構造的に固定されます。
  起点画像は「キャラ1体だけ・文字や枠線が入っていないもの」にしてください。
  （キャラクター設定シートを起点にすると、見出しの文字が最後まで焼き付きます）

● VSAが効いているかログで確認してください
  [sol_attn] producer path: ... VSA tiles ... topk=0.100
  この行が出ていなければ、間引きが効かず通常計算に落ちています。
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

# ========================================
# 🆕 サンプル画像の自動ダウンロード（inputフォルダへ格納）
# ========================================
print("\n📥 Downloading sample image to input folder...")
input_dir = "/content/ComfyUI/input"

# I2V用のサンプル画像
model_download("https://github.com/aicuai/Book-SG26/blob/main/WebUI_Launch_Setup_Files/sg26_I2V_image_SAMPLE_mei.png", input_dir)
model_download("https://github.com/aicuai/Book-SG26/blob/main/WebUI_Launch_Setup_Files/sg26_I2V_image_SAMPLE_rhyth.png", input_dir)

# FLF2V用のサンプル画像（全6点）
model_download("https://github.com/aicuai/Book-SG26/blob/main/WebUI_Launch_Setup_Files/sg26_FLF2V_image_SAMPLE_mei_01.png", input_dir)
model_download("https://github.com/aicuai/Book-SG26/blob/main/WebUI_Launch_Setup_Files/sg26_FLF2V_image_SAMPLE_mei_02.png", input_dir)
model_download("https://github.com/aicuai/Book-SG26/blob/main/WebUI_Launch_Setup_Files/sg26_FLF2V_image_SAMPLE_mei_chibi.png", input_dir)
model_download("https://github.com/aicuai/Book-SG26/blob/main/WebUI_Launch_Setup_Files/sg26_FLF2V_image_SAMPLE_mei_logo_01.png", input_dir)
model_download("https://github.com/aicuai/Book-SG26/blob/main/WebUI_Launch_Setup_Files/sg26_FLF2V_image_SAMPLE_mei_logo_02.png", input_dir)
model_download("https://github.com/aicuai/Book-SG26/blob/main/WebUI_Launch_Setup_Files/sg26_FLF2V_image_SAMPLE_rhyth_chibi.png", input_dir)

print("✓ All sample images downloaded!\n")



# ========================================
# ComfyUI起動設定
# ========================================

# 🆕 出力ディレクトリの設定
output_dir_arg = ""
if use_google_drive and enable_gdrive_output:
    output_dir_arg = f"--output-directory {GDRIVE_OUTPUT}"
    print("=" * 70)
    print(f"✅ ComfyUI output will be saved directly to Google Drive")
    print(f"📁 Output path: {GDRIVE_OUTPUT}")
    print("=" * 70)

print("🚀 Pinggy Tunnel を準備しています...")
import subprocess
import threading
import time
import socket
import urllib.request
import html
import re
from IPython.display import HTML, display

def iframe_thread(port):
    while True:
        time.sleep(0.5)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        if result == 0:
            break
        sock.close()
    print("\nComfyUI finished loading... setting up Pinggy tunnel!\n")
    print("(Pinggyは、Colab上のComfyUIを外部から使えるようにする一時トンネルサービスです。下に出るURLはこのセッション限りの一時URLです)\n")

    # ★ Pinggy有料トークンをColabシークレットから取得
    # Colab左側の「シークレット」に登録している名前に合わせています。
    # バックグラウンドスレッドからのシークレット取得はタイムアウトしやすいため、リトライする。
    from google.colab import userdata
    PINGGY_TOKEN = None
    for _pinggy_attempt in range(5):
        try:
            PINGGY_TOKEN = userdata.get("PINGGY_TOKEN")
            break
        except Exception as _pinggy_err:
            print(f"⚠️ PINGGY_TOKENの取得に失敗（{_pinggy_attempt + 1}/5回目）: {_pinggy_err}")
            time.sleep(3)

    if not PINGGY_TOKEN:
        raise ValueError(
            "Colabのシークレットに PINGGY_TOKEN が登録されていないか、5回リトライしても取得できませんでした。\n"
            "左側の🔑シークレットで名前とノートブックからのアクセス権（ON）を確認し、\n"
            "改善しない場合はこのセルを一度停止してから再実行してください。"
        )

    # ★ 有料Pinggy用に変更
    # 変更前: a.pinggy.io
    # 変更後: <Pinggyトークン>@pro.pinggy.io
    p = subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-o", "ServerAliveInterval=60", "-p", "443", "-R0:localhost:{}".format(port), f"{PINGGY_TOKEN}@pro.pinggy.io"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    for line in p.stdout:
        l = line.decode()
        match = re.search(r"https://[^\s]+", l)
        if match and "dashboard.pinggy.io" not in match.group(0):
            comfyui_url = match.group(0).rstrip(".,;)]}")
            print("\n" + "="*70)
            print("🚀 Pinggy URL:", comfyui_url)
            print("="*70 + "\n")
            safe_url = html.escape(comfyui_url, quote=True)
            display(HTML(
                f'<a href="{safe_url}" target="_blank" '
                'style="display:inline-block;padding:12px 20px;background:#1976d2;color:white;'
                'font-weight:bold;text-decoration:none;border-radius:8px">ComfyUIを開く</a>'
            ))
            print("✅URLからリンクを開いたら Pinggy の赤色の🟥「Enter Site」のボタンをクリックして下さい。")
            print("有料トークンで接続しているため、無料枠の60分制限ではなくPinggy Proの設定で動作します。")
            break

clear_output()
threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ★ 最重要修正箇所: --listen 0.0.0.0 と --enable-cors-header を追加して 403 Forbidden を回避します
# @markdown ---
# @markdown ## ⚡ Sage Attention（上級者向け、既定OFF推奨）
# @markdown 🌱FastH3はVSA(Sol-Attn)で高速化しているため、Sage Attentionは不要です。**必ずOFFのままにしてください**（併用すると衝突します）。
use_sage_attention = False  # @param {type:"boolean"}

if use_sage_attention:
    try:
        get_ipython().system("pip install -q sageattention")
        print("✅ sageattentionのインストールを試みました（失敗していてもComfyUIは起動します）")
    except Exception as _sage_install_err:
        print(f"⚠️ sageattentionのインストールに失敗しました（続行します）: {_sage_install_err}")
    _sage_flag = " --use-sage-attention"
else:
    print("ℹ️ Sage Attentionは無効です（use_sage_attention=Falseのため）")
    _sage_flag = ""

if using_L4_GPU:
    command = f"python main.py {output_dir_arg} --listen 0.0.0.0 --enable-cors-header --cache-none{_sage_flag} --dont-print-server".strip()
    get_ipython().system(command)
else:
    command = f"python main.py {output_dir_arg} --listen 0.0.0.0 --enable-cors-header{_sage_flag} --dont-print-server".strip()
    get_ipython().system(command)



# 🌱📋MiniMax H3用モデル_DL元リポジトリのメモ🌱
### 下記はMiniMax H3専用の構成です。上のセルの既定値は、この表の①②③の組み合わせ（R2V＋FL2VA＋Qwenテキストエンコーダー＋VAE2種）を使い、T2V/I2V/R2Vのいずれも試せる状態にしています。
---
#🎬MiniMax H3 本体（ComfyUI公式量子化配布：Comfy-Org/MiniMax-H3）
##①＜拡散モデル R2V用＞ ※参照画像・参照動画・参照音声から動画を作る。キャラシートなどの参照素材を使う用途はこちら
- minimax_h3_ref2va_pruned_int8_convrot.safetensors（約19.5GiB）：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_pruned_int8_convrot.safetensors)

##＜拡散モデル T2V/I2V用＞ ※テキストのみ、または開始・終了画像から動画を作る
- minimax_h3_fl2va_pruned_int8_convrot.safetensors（約19.5GiB）：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors)

##＜非pruned・高品質比較用（任意・容量大）＞
- minimax_h3_ref2va_int8_convrot.safetensors：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_ref2va_int8_convrot.safetensors)
- minimax_h3_fl2va_int8_convrot.safetensors：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_int8_convrot.safetensors)
- minimax_h3_ref2va_bf16.safetensors / minimax_h3_fl2va_bf16.safetensors：フルBF16。4090・A100 80GB単体では非推奨（ComfyUI公式ブログでフル精度は約123.6GBと説明）

##②＜テキストエンコーダー＞ ※必須。Qwen3-VL-32Bベースで、プロンプトと参照画像・参照動画を理解する
- qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors（約14.6GiB・既定）：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors)
- qwen3vl_32b_minimax_h3_int8_convrot.safetensors（品質重視・容量大）：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_int8_convrot.safetensors)
- qwen3vl_32b_minimax_h3_bf16.safetensors（フルBF16・非推奨）：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_bf16.safetensors)

##③＜VAE＞ ※必須。H3は動画と音声を同時に扱うため両方入れる
- minimax_h3_video_vae_fp16.safetensors（約4.85GiB）：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors)
- minimax_h3_audio_vae_fp32.safetensors（約0.56GiB）：(https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors)

#🧩必須カスタムノード
- **不要**。MiniMax H3ノードはComfyUI公式PR #15224でComfyUI本体（v0.30.0以降）へマージ済み。個別のcustom_nodesリポジトリをgit cloneする必要はない。
- 起動セルはComfyUIをgit clone後、`git pull`でmainブランチを最新化し、`comfyui_version.py`でv0.30.0以降であることを自動確認する。

#📄公式ワークフローテンプレート（自動配置済み）
起動セルが以下をGitHubから取得し、`ComfyUI/user/default/workflows/MiniMax_H3/`へ保存する。ComfyUI起動後、左メニューのWorkflows（User workflows）から選ぶ。
- video_minimax_h3_t2v.json：(https://github.com/Comfy-Org/workflow_templates/blob/main/templates/video_minimax_h3_t2v.json)
- video_minimax_h3_i2v.json：(https://github.com/Comfy-Org/workflow_templates/blob/main/templates/video_minimax_h3_i2v.json)
- video_minimax_h3_r2v.json：(https://github.com/Comfy-Org/workflow_templates/blob/main/templates/video_minimax_h3_r2v.json)

#⚠️GPU・容量の目安（2026-08-05更新）
- 既定構成（Ref2VA＋FL2VA＋Qwen NVFP4-AWQ＋VAE2種）の保存容量は合計約59GiB。R2Vだけに絞るならFL2VAを削って約44GiBに減らせる。**これらはGoogle Driveに保存され、2回目以降は再ダウンロードされない。**
- 公式の最低/推奨VRAMは非公開。目安：24GB（RTX4090・L4級）はR2V短尺・低〜中解像度の実験可能性あり、48GB以上は768p・複数参照・音声付きで余裕、80GB（A100）が最も現実的な検証先。
- ColabのL4（24GB）は、うちのローカルRTX4090（24GB）での実測（EasyCache＋Sage Attention、0.9メガピクセル・15秒R2Vが約12〜13分）に近い挙動が期待できる。快適な長尺・2K運用は依然として期待しない方がよい。
- H3-Regenerate-2K（768p→2K再生成）とH3-Context-IRはローカル非公開。ローカルだけで公式2K工程は完結できない。
- 💾**モデルの保存先はGoogle Drive**（既定: `/content/drive/MyDrive/MiniMaxH3_Models/`）。Driveの空き容量は最低60GB程度を確保しておくこと。

#⚠️ライセンス上の注意
- MiniMax Community Licenseの適用地域に日本は含まれるが、米国・EU・英国・韓国は除外地域。
- 商用サービスの年間売上が2,000万米ドル超の場合は事前の書面許可が必要。商用UIには`MiniMax H3`の表示が必要。
- 詳細：(https://huggingface.co/MiniMaxAI/MiniMax-H3/blob/main/LICENSE)

#📌参照素材の入力条件（R2V使用時の目安）
- 画像：JPG/JPEG/PNG/WEBP/HEIC/HEIF、1枚30MB以下、最大9枚
- 動画：MP4/MOV、1本50MB以下・2〜15秒・合計15秒以下・最大3本
- 音声：WAV/MP3、1本15MB以下・2〜15秒・合計15秒以下・最大3本（音声のみの参照は不可。画像または動画の参照が最低1つ必要）

#🔗参考出典
- ComfyUI公式チュートリアル：(https://docs.comfy.org/tutorials/video/minimax/minimax-h3)
- ComfyUI公式ブログ（day-0対応・約42.5GB構成・約66%削減）：(https://blog.comfy.org/p/minimax-h3-day-0-support-in-comfyui)
- ComfyUI H3実装PR #15224：(https://github.com/Comfy-Org/ComfyUI/pull/15224)
- Comfy-Org配布先：(https://huggingface.co/Comfy-Org/MiniMax-H3)
- MiniMax公式フルモデル：(https://huggingface.co/MiniMaxAI/MiniMax-H3)
- 詳細ナレッジ：`B:\Codex_Workspace\geneko-desktop-assistant-mvp\data\knowledge\glossary\ai-tools\minimax-h3.md`
---


---

## 🌱 FastH3版で使うモデル

| フォルダ | ファイル | サイズ | 配布元 |
| --- | --- | --- | --- |
| `diffusion_models` | `minimax_h3_fastvideo_vsa_datafree_1300step_4step_int8_convrot.safetensors` | 約21.3GiB | [Kijai/MiniMax-H3-experimental](https://huggingface.co/Kijai/MiniMax-H3-experimental) |
| `text_encoders` | `qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors` | 約14.6GiB | [Comfy-Org/MiniMax-H3](https://huggingface.co/Comfy-Org/MiniMax-H3) |
| `vae` | `minimax_h3_video_vae_fp16.safetensors` | 約4.5GiB | 同上 |
| `vae` | `minimax_h3_audio_vae_fp32.safetensors` | 約0.5GiB | 同上 |

**合計およそ 41GiB** です。Drive常駐版を使う場合は、Google ドライブに **44GB以上の空き**が必要です（無料枠15GBでは足りません）。

### モデル以外に取得するもの

| 何 | どこから | 再配布 |
| --- | --- | --- |
| ComfyUI（VSA対応ブランチ） | [kijai/ComfyUI `vsa`](https://github.com/kijai/ComfyUI/tree/vsa) | しない |
| comfy-kitchen（sol_attn入り） | 本Notebookの配布リポジトリのReleases | **Apache-2.0のため再配布可** |
| `sol_attn_minimax_v5.py` | [comfy-kitchen PR #117 の添付](https://github.com/Comfy-Org/comfy-kitchen/pull/117) | しない |
| サンプルワークフロー | [sepiablue-ai/minimax_h3_workflows](https://github.com/sepiablue-ai/minimax_h3_workflows) | しない |

**FastH3の重み・暫定ノード・ワークフローJSONにはライセンス表記がありません。**
使うのは問題ありませんが、**再配布はしないでください**（本Notebookも取得先へのリンクだけを持っています）。
